# Figure 5 | Anatomical gene expression gradients in HD

1,287-gene anatomical heatmaps, category profiles, GO score heatmaps, and a reusable spatial-map routine. Exact configurations for some panels are not included.

Check input/output paths and settings before running each workflow. Run its cells in order with a fresh kernel. See README.md for required inputs, panel coverage, and configuration differences.

| Panels | Workflow |
|---|---|
| B, H, N | Reusable spatial-map routine; final panel calls absent |
| C, E, I, K, O | WT-ordered gene heatmaps |
| D, F, J, L, P | WT-defined category profiles |
| Q | Relative GO score heatmaps |
| A, G, M | Schematics |


## Panels B, H, N | Reusable single-gene spatial maps

Exact 1,287-gene spatial-map calls are not included. The example uses 191-gene data and does not reproduce these panels with its defaults. Panel-specific inputs, annotation columns, coordinate flips, genes, regions, and spatial-Z expression settings require confirmation against the 1,287-gene data. Panel genes are Coch/Galntl6/Lamp5/Oprm1 (dorsal striatum), Galntl6/Erbb4/Htr4/Tshz2 (NAcc), and Cnr1/Fign/Sema5b (OT).


In [ ]:
# Imports and settings.
from pathlib import Path
import re
import pandas as pd
import scanpy as sc


H5AD_FILE = Path(r"F:\BASSFISH_HD\Source_data\integrated_5_str_with_sample_level_per_cell_layers.h5ad")

OUT_DIR = Path(r"F:\BASSFISH_HD\Figure\Figure_3\Panel\show_single_gene")


GENES = ["Tshz1", "Sgk1"]


REGIONS = ["d"]


LAYER = "raw"
SAMPLE_COL = "sample"
CLUSTER_COL = "refined_cluster_6_merged"
REGION_COL = "region_str_manual"
X_COL = "x"
Y_COL = "y"


CLUSTER_SETS = {
    "D1": "MSN_D1",
    "D2": "MSN_D2",
    "D1_plus_D2": ["MSN_D1", "MSN_D2"],
}


FLIP_ORIGINAL_SAMPLE_COORDINATES = True
SAMPLES_FLIP_X = ["Homo_3", "WT_5"]
SAMPLES_FLIP_Y = ["Homo_4", "WT_2", "WT_3", "WT_4"]


BIN_SIZE = 124
MIN_CELLS_PER_BIN = 1
SMOOTH_SIGMA = 1
SMOOTH_EXPAND = True
SMOOTH_WEIGHT_THRESHOLD = 0.05


DEFAULT_COLOR_LIMIT = 3
PER_GENE_COLOR_LIMIT = {


}
CMAP = "RdBu_r"


NCOLS = 4
PANEL_WIDTH_MM = 12
SHOW_SAMPLE_TITLE = True
SHOW_CELL_NUMBER = True
SHOW_SCALE_BAR = True
SCALE_BAR_UM = 500
UM_PER_COORDINATE = 0.16
SCALE_BAR_EACH_PANEL = True
SHOW_COLORBAR = True
COLORBAR_WIDTH_MM = 1.2
COLORBAR_HEIGHT_MM = 18
COLORBAR_LABEL = "Spatial z-score"
SAVE_DPI = 600


SHOW_FIGURES = False


In [ ]:
if not H5AD_FILE.is_file():
    raise FileNotFoundError(f"h5ad file not found: {H5AD_FILE}")

adata = sc.read_h5ad(H5AD_FILE)
print(adata)

for col in [SAMPLE_COL, CLUSTER_COL, REGION_COL, X_COL, Y_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing obs[{col!r}]; available columns: {list(adata.obs.columns)}")

if LAYER not in adata.layers:
    raise KeyError(f"Missing layers[{LAYER!r}]; available layers: {list(adata.layers.keys())}")

if FLIP_ORIGINAL_SAMPLE_COORDINATES:
    adata.obs[X_COL + "_original"] = adata.obs[X_COL].copy()
    adata.obs[Y_COL + "_original"] = adata.obs[Y_COL].copy()

    sample_id = adata.obs[SAMPLE_COL].astype(str)
    adata.obs.loc[sample_id.isin(SAMPLES_FLIP_X), X_COL] *= -1
    adata.obs.loc[sample_id.isin(SAMPLES_FLIP_Y), Y_COL] *= -1
    print("Applied the configured sample-specific x/y coordinate flips in memory.")

clusters_available = set(adata.obs[CLUSTER_COL].astype(str).unique())
for required_cluster in ["MSN_D1", "MSN_D2"]:
    if required_cluster not in clusters_available:
        raise ValueError(
            f"obs[{CLUSTER_COL!r}] does not contain {required_cluster}.\n"
            f"Available categories: {sorted(clusters_available)}"
        )

if GENES is None:
    genes_to_plot = adata.var_names.astype(str).tolist()
else:
    genes_to_plot = list(dict.fromkeys(str(g) for g in GENES))

missing_genes = [g for g in genes_to_plot if g not in adata.var_names]
if missing_genes:
    raise ValueError(f"Genes missing from adata.var_names: {missing_genes}")

if not genes_to_plot:
    raise ValueError("GENES is empty.")

for region in REGIONS:
    if str(region) not in set(adata.obs[REGION_COL].astype(str)):
        raise ValueError(f"Region {region!r} is absent from obs[{REGION_COL!r}]")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plotting {len(genes_to_plot)} genes x {len(REGIONS)} regions x 3 cell-type combinations.")


In [ ]:
# Imports and settings.
import math
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.transforms import blended_transform_factory
from scipy import sparse
from scipy.ndimage import gaussian_filter


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 7,

    "axes.titlesize": 7,
    "axes.labelsize": 7,

    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    "legend.fontsize": 7,
    "legend.title_fontsize": 7,

    "figure.titlesize": 7,


    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.unicode_minus": False,
})


def _to_list(value):
    """Convert one value or a sequence to a list."""


    if value is None:
        return None

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return list(value)

    return [value]


def _add_scale_bar(
    ax,
    length_um=500,
    um_per_coordinate=0.16,
    color="black",
    linewidth=0.8,
    fontsize=7,
    x_pad_fraction=0.06,
    y_position_fraction=0.07,
    text_gap_fraction=0.025,
):
    """Add a scale bar to the spatial plot."""


    if length_um is None:
        return

    if length_um <= 0:
        raise ValueError("length_um must be greater than 0")

    if um_per_coordinate <= 0:
        raise ValueError("um_per_coordinate must be greater than 0")


    length_coordinate = length_um / um_per_coordinate

    x_min, x_max = sorted(ax.get_xlim())
    x_range = x_max - x_min

    if x_range <= 0:
        return

    if length_coordinate >= x_range:
        raise ValueError(
            f"Scale bar too long: {length_um} micrometers corresponds to "
            f"{length_coordinate:.1f} coordinate units; "
            f"the subplot width is only {x_range:.1f} coordinate units."
        )

    x_right = x_max - x_pad_fraction * x_range
    x_left = x_right - length_coordinate
    x_center = (x_left + x_right) / 2


    transform = blended_transform_factory(
        ax.transData,
        ax.transAxes,
    )


    ax.plot(
        [x_left, x_right],
        [y_position_fraction, y_position_fraction],
        transform=transform,
        color=color,
        linewidth=linewidth,
        solid_capstyle="butt",
        clip_on=False,
        zorder=100,
    )


    ax.text(
        x_center,
        y_position_fraction + text_gap_fraction,
        f"{length_um:g} µm",
        transform=transform,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontfamily="Arial",
        fontweight="normal",
        color=color,
        clip_on=False,
        zorder=100,
    )


def plot_spatial_heatmap_all_samples(
    adata,
    gene,
    clusters=None,
    regions=None,


    layer="spatial_z_cell",
    sample_col="sample",
    cluster_col="refined_cluster_6_merged",
    region_col="region_str_manual",
    x_col="x",
    y_col="y",


    bin_size=50,
    min_cells_per_bin=1,
    smooth_sigma=1,
    smooth_expand=True,
    smooth_weight_threshold=0.05,


    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    percentile=99,


    ncols=4,
    panel_width_mm=12,
    invert_y=True,
    use_full_sample_extent=True,
    wspace=0.12,
    hspace=0.30,


    title="",
    show_sample_title=True,
    show_cell_number=True,


    show_colorbar=True,
    colorbar_label=None,
    colorbar_width_mm=1.2,
    colorbar_height_mm=18,
    colorbar_gap_mm=2,


    show_scale_bar=True,
    scale_bar_um=500,
    um_per_coordinate=0.16,
    scale_bar_color="black",
    scale_bar_linewidth=0.8,
    scale_bar_each_panel=True,


    save_path=None,
    save_dpi=600,
    show=True,
):
    """Plot sample-wise spatial expression heatmaps."""


    font_size = 7
    mm_to_inch = 1 / 25.4


    if gene not in adata.var_names:
        raise ValueError(
            f"Gene {gene!r} is missing from adata.var_names"
        )

    if layer not in adata.layers:
        raise ValueError(
            f"Layer {layer!r} is missing from adata.layers"
        )

    required_cols = [
        sample_col,
        cluster_col,
        region_col,
        x_col,
        y_col,
    ]

    for col in required_cols:
        if col not in adata.obs.columns:
            raise ValueError(
                f"Column {col!r} is missing from adata.obs"
            )

    if bin_size <= 0:
        raise ValueError("bin_size must be greater than 0")

    if min_cells_per_bin < 1:
        raise ValueError(
            "min_cells_per_bin must be at least 1"
        )

    if smooth_sigma < 0:
        raise ValueError(
            "smooth_sigma must be nonnegative"
        )

    if ncols < 1:
        raise ValueError("ncols must be at least 1")

    if panel_width_mm <= 0:
        raise ValueError(
            "panel_width_mm must be greater than 0"
        )


    clusters_list = _to_list(clusters)

    if clusters_list is None:
        cluster_mask = np.ones(
            adata.n_obs,
            dtype=bool,
        )
        cluster_title = "all"

    else:
        clusters_str = [
            str(value)
            for value in clusters_list
        ]

        cluster_mask = (
            adata.obs[cluster_col]
            .astype(str)
            .isin(clusters_str)
            .to_numpy()
        )

        cluster_title = ", ".join(clusters_str)


    regions_list = _to_list(regions)

    if regions_list is None:
        region_mask = np.ones(
            adata.n_obs,
            dtype=bool,
        )
        region_title = "all"

    else:
        regions_str = [
            str(value)
            for value in regions_list
        ]

        region_mask = (
            adata.obs[region_col]
            .astype(str)
            .isin(regions_str)
            .to_numpy()
        )

        region_title = ", ".join(regions_str)

    selected_mask = cluster_mask & region_mask

    if selected_mask.sum() == 0:
        raise ValueError(
            "No cells remain after filtering.\n"
            f"{cluster_col}={clusters}\n"
            f"{region_col}={regions}"
        )


    values_all = adata[:, gene].layers[layer]

    if sparse.issparse(values_all):
        values_all = values_all.toarray()

    values_all = (
        np.asarray(values_all)
        .ravel()
        .astype(float)
    )

    x_all = adata.obs[x_col].to_numpy(
        dtype=float
    )

    y_all = adata.obs[y_col].to_numpy(
        dtype=float
    )

    sample_values = (
        adata.obs[sample_col]
        .astype(str)
        .to_numpy()
    )

    samples = (
        adata.obs.loc[selected_mask, sample_col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    samples = sorted(samples)

    if len(samples) == 0:
        raise ValueError(
            "No samples remain for plotting after filtering"
        )


    results = []
    all_heatmap_values = []

    for sample_name in samples:

        sample_mask = (
            sample_values == sample_name
        )

        plot_mask = (
            selected_mask & sample_mask
        )

        x = x_all[plot_mask]
        y = y_all[plot_mask]
        z = values_all[plot_mask]

        valid = (
            np.isfinite(x)
            & np.isfinite(y)
            & np.isfinite(z)
        )

        x = x[valid]
        y = y[valid]
        z = z[valid]

        if len(x) == 0:
            results.append(None)
            continue


        if use_full_sample_extent:

            x_reference = x_all[sample_mask]
            y_reference = y_all[sample_mask]

            reference_valid = (
                np.isfinite(x_reference)
                & np.isfinite(y_reference)
            )

            x_reference = x_reference[
                reference_valid
            ]

            y_reference = y_reference[
                reference_valid
            ]

        else:
            x_reference = x
            y_reference = y

        if len(x_reference) == 0:
            results.append(None)
            continue

        x_min = np.nanmin(x_reference)
        x_max = np.nanmax(x_reference)
        y_min = np.nanmin(y_reference)
        y_max = np.nanmax(y_reference)

        if x_max == x_min:
            x_max = x_min + bin_size

        if y_max == y_min:
            y_max = y_min + bin_size


        x_start = (
            np.floor(x_min / bin_size)
            * bin_size
        )

        x_end = (
            np.ceil(x_max / bin_size)
            * bin_size
        )

        y_start = (
            np.floor(y_min / bin_size)
            * bin_size
        )

        y_end = (
            np.ceil(y_max / bin_size)
            * bin_size
        )

        x_edges = np.arange(
            x_start,
            x_end + bin_size,
            bin_size,
        )

        y_edges = np.arange(
            y_start,
            y_end + bin_size,
            bin_size,
        )

        if len(x_edges) < 2:
            x_edges = np.array([
                x_start,
                x_start + bin_size,
            ])

        if len(y_edges) < 2:
            y_edges = np.array([
                y_start,
                y_start + bin_size,
            ])


        count_map, _, _ = np.histogram2d(
            y,
            x,
            bins=[y_edges, x_edges],
        )


        sum_map, _, _ = np.histogram2d(
            y,
            x,
            bins=[y_edges, x_edges],
            weights=z,
        )


        with np.errstate(
            divide="ignore",
            invalid="ignore",
        ):
            mean_map = sum_map / count_map

        occupied_mask = (
            count_map >= min_cells_per_bin
        )

        mean_map[~occupied_mask] = np.nan


        if smooth_sigma > 0:

            finite_mask = np.isfinite(
                mean_map
            )

            value_filled = np.nan_to_num(
                mean_map,
                nan=0.0,
            )

            weight_map = finite_mask.astype(
                float
            )

            smooth_value = gaussian_filter(
                value_filled * weight_map,
                sigma=smooth_sigma,
                mode="constant",
                cval=0,
            )

            smooth_weight = gaussian_filter(
                weight_map,
                sigma=smooth_sigma,
                mode="constant",
                cval=0,
            )

            with np.errstate(
                divide="ignore",
                invalid="ignore",
            ):
                smooth_map = (
                    smooth_value / smooth_weight
                )

            if smooth_expand:

                smooth_map[
                    smooth_weight
                    < smooth_weight_threshold
                ] = np.nan

            else:

                smooth_map[
                    ~occupied_mask
                ] = np.nan

            mean_map = smooth_map

        valid_heatmap = mean_map[
            np.isfinite(mean_map)
        ]

        if valid_heatmap.size > 0:
            all_heatmap_values.append(
                valid_heatmap
            )

        results.append({
            "sample": sample_name,
            "map": mean_map,
            "count_map": count_map,
            "x_edges": x_edges,
            "y_edges": y_edges,
            "n_cells": int(plot_mask.sum()),
        })

    if len(all_heatmap_values) == 0:
        raise ValueError(
            "No samples have valid heatmaps.\n"
            "Increase bin_size, decrease min_cells_per_bin, "
            "or check the filtering criteria."
        )

    all_heatmap_values = np.concatenate(
        all_heatmap_values
    )


    if vmin is None and vmax is None:

        vmax_plot = np.nanpercentile(
            np.abs(all_heatmap_values),
            percentile,
        )

        if (
            not np.isfinite(vmax_plot)
            or vmax_plot == 0
        ):
            vmax_plot = 1.0

        vmin_plot = -vmax_plot

    elif vmin is not None and vmax is not None:

        vmin_plot = float(vmin)
        vmax_plot = float(vmax)

    elif vmin is None:

        vmax_plot = float(vmax)
        vmin_plot = -abs(vmax_plot)

    else:

        vmin_plot = float(vmin)
        vmax_plot = abs(vmin_plot)

    if vmin_plot >= vmax_plot:
        raise ValueError(
            f"Invalid color limits: "
            f"vmin={vmin_plot}, "
            f"vmax={vmax_plot}"
        )


    nrows = math.ceil(
        len(samples) / ncols
    )

    panel_width_inch = (
        panel_width_mm * mm_to_inch
    )


    aspect_ratios = []

    for result in results:

        if result is None:
            continue

        x_range = (
            result["x_edges"][-1]
            - result["x_edges"][0]
        )

        y_range = (
            result["y_edges"][-1]
            - result["y_edges"][0]
        )

        if x_range > 0 and y_range > 0:
            aspect_ratios.append(
                y_range / x_range
            )

    max_aspect_ratio = (
        max(aspect_ratios)
        if len(aspect_ratios) > 0
        else 1.0
    )


    plot_height_inch = (
        panel_width_inch
        * max_aspect_ratio
    )


    title_extra_mm = (
        4 if show_sample_title else 1
    )

    row_slot_height_inch = (
        plot_height_inch
        + title_extra_mm * mm_to_inch
    )


    left = 0.03
    right = 0.84 if show_colorbar else 0.98
    bottom = 0.03

    if title not in (None, ""):
        top = 0.91
    else:
        top = 0.97


    figure_width_inch = (
        panel_width_inch
        * (
            ncols
            + (ncols - 1) * wspace
        )
        / (right - left)
    )


    figure_height_inch = (
        row_slot_height_inch
        * (
            nrows
            + (nrows - 1) * hspace
        )
        / (top - bottom)
    )

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(
            figure_width_inch,
            figure_height_inch,
        ),
        squeeze=False,
        constrained_layout=False,
    )

    fig.subplots_adjust(
        left=left,
        right=right,
        bottom=bottom,
        top=top,
        wspace=wspace,
        hspace=hspace,
    )

    mesh = None


    for panel_index, (ax, result) in enumerate(
        zip(axes.flat, results)
    ):

        if result is None:
            ax.axis("off")
            continue

        heatmap_masked = np.ma.masked_invalid(
            result["map"]
        )

        mesh = ax.pcolormesh(
            result["x_edges"],
            result["y_edges"],
            heatmap_masked,
            cmap=cmap,
            vmin=vmin_plot,
            vmax=vmax_plot,
            shading="flat",
            edgecolors="none",
            linewidth=0,
            antialiased=False,


            rasterized=True,
        )

        ax.set_xlim(
            result["x_edges"][0],
            result["x_edges"][-1],
        )

        ax.set_ylim(
            result["y_edges"][0],
            result["y_edges"][-1],
        )

        ax.set_aspect(
            "equal",
            adjustable="box",
        )

        if invert_y:
            ax.invert_yaxis()


        if show_sample_title:
            sample_title = result["sample"]

            ax.set_title(
                sample_title,
                fontsize=font_size,
                fontfamily="Arial",
                fontweight="normal",
                pad=1.5,
            )

        ax.axis("off")


        if show_scale_bar:

            if (
                scale_bar_each_panel
                or panel_index == 0
            ):

                _add_scale_bar(
                    ax=ax,
                    length_um=scale_bar_um,
                    um_per_coordinate=(
                        um_per_coordinate
                    ),
                    color=scale_bar_color,
                    linewidth=(
                        scale_bar_linewidth
                    ),
                    fontsize=font_size,
                )


    for ax in axes.flat[len(samples):]:
        ax.axis("off")


    if show_colorbar and mesh is not None:

        colorbar_width_inch = (
            colorbar_width_mm
            * mm_to_inch
        )

        colorbar_height_inch = (
            colorbar_height_mm
            * mm_to_inch
        )

        colorbar_gap_inch = (
            colorbar_gap_mm
            * mm_to_inch
        )

        colorbar_left_inch = (
            figure_width_inch * right
            + colorbar_gap_inch
        )

        colorbar_bottom_inch = (
            figure_height_inch
            - colorbar_height_inch
        ) / 2

        cax = fig.add_axes([
            colorbar_left_inch
            / figure_width_inch,

            colorbar_bottom_inch
            / figure_height_inch,

            colorbar_width_inch
            / figure_width_inch,

            colorbar_height_inch
            / figure_height_inch,
        ])

        cbar = fig.colorbar(
            mesh,
            cax=cax,
        )

        if colorbar_label is None:
            colorbar_label = gene

        cbar.set_label(
            colorbar_label,
            rotation=90,
            labelpad=4,
            fontsize=font_size,
            fontfamily="Arial",
            fontweight="normal",
        )

        cbar.ax.tick_params(
            labelsize=font_size,
            length=2,
            width=0.5,
        )

        for tick_label in (
            cbar.ax.get_yticklabels()
        ):
            tick_label.set_fontfamily(
                "Arial"
            )
            tick_label.set_fontsize(
                font_size
            )

        cbar.outline.set_linewidth(0.5)


    if title is None:
        title = (
            f"{gene} | "
            f"{cluster_col}: {cluster_title} | "
            f"{region_col}: {region_title}"
        )

    if title != "":
        fig.suptitle(
            title,
            fontsize=font_size,
            fontfamily="Arial",
            fontweight="normal",
            y=0.98,
        )


    if save_path is not None:

        save_path = Path(save_path)

        if save_path.suffix.lower() != ".pdf":
            save_path = (
                save_path.with_suffix(".pdf")
            )

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            save_path,
            format="pdf",


            dpi=save_dpi,


            bbox_inches=None,
            pad_inches=0,

            facecolor="white",
            transparent=False,
        )

        print(
            f"PDF saved: "
            f"{save_path.resolve()}"
        )

        print(
            f"PDF page dimensions: "
            f"{figure_width_inch * 25.4:.2f} × "
            f"{figure_height_inch * 25.4:.2f} mm"
        )

        print(
            f"Target width of each spatial plot: "
            f"{panel_width_mm:.2f} mm"
        )


    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes, results


In [ ]:
def safe_filename(value):
    return re.sub(r'[<>:"/\\|?*\s]+', "_", str(value)).strip("._") or "unnamed"


manifest = []
for region in REGIONS:
    for gene in genes_to_plot:
        limit = float(PER_GENE_COLOR_LIMIT.get(gene, DEFAULT_COLOR_LIMIT))
        if limit <= 0:
            raise ValueError(f"{gene}: color limit must be greater than 0")

        for mode, cluster_selection in CLUSTER_SETS.items():
            output_pdf = (
                OUT_DIR
                / f"{safe_filename(gene)}__{safe_filename(region)}__{mode}__{safe_filename(LAYER)}.pdf"
            )
            print(f"\n>>> {gene} | region={region} | {mode}")

            fig, axes, sample_results = plot_spatial_heatmap_all_samples(
                adata=adata,
                gene=gene,
                layer=LAYER,
                clusters=cluster_selection,
                regions=region,
                sample_col=SAMPLE_COL,
                cluster_col=CLUSTER_COL,
                region_col=REGION_COL,
                x_col=X_COL,
                y_col=Y_COL,
                bin_size=BIN_SIZE,
                min_cells_per_bin=MIN_CELLS_PER_BIN,
                smooth_sigma=SMOOTH_SIGMA,
                smooth_expand=SMOOTH_EXPAND,
                smooth_weight_threshold=SMOOTH_WEIGHT_THRESHOLD,
                cmap=CMAP,
                vmin=-limit,
                vmax=limit,
                ncols=NCOLS,
                panel_width_mm=PANEL_WIDTH_MM,
                invert_y=True,
                use_full_sample_extent=True,
                show_sample_title=SHOW_SAMPLE_TITLE,
                show_cell_number=SHOW_CELL_NUMBER,
                show_scale_bar=SHOW_SCALE_BAR,
                scale_bar_um=SCALE_BAR_UM,
                um_per_coordinate=UM_PER_COORDINATE,
                scale_bar_color="black",
                scale_bar_linewidth=0.8,
                scale_bar_each_panel=SCALE_BAR_EACH_PANEL,
                show_colorbar=SHOW_COLORBAR,
                colorbar_label=COLORBAR_LABEL,
                colorbar_width_mm=COLORBAR_WIDTH_MM,
                colorbar_height_mm=COLORBAR_HEIGHT_MM,
                title="",
                save_path=output_pdf,
                save_dpi=SAVE_DPI,
                show=SHOW_FIGURES,
            )

            manifest.append({
                "gene": gene,
                "region": str(region),
                "mode": mode,
                "clusters": str(cluster_selection),
                "layer": LAYER,
                "vmin": -limit,
                "vmax": limit,
                "n_samples": sum(r is not None for r in sample_results),
                "pdf": str(output_pdf),
            })

manifest_df = pd.DataFrame(manifest)
manifest_path = OUT_DIR / "D1_D2_combined_spatial_heatmap_manifest.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")
print(f"\nCompleted: {len(manifest_df)} PDF files")
print(f"Output directory: {OUT_DIR}")
print(f"Output manifest: {manifest_path}")


## Panels C, E, I, K, O | WT-ordered anatomical heatmaps

The default `HEATMAP_COLOR_MODE="relative_mean"` differs from the manuscript's row Z scores. The plotting function also supports `row_zscore`. The batch processes D1 and D2; Figure 5 uses D1 outputs.


In [ ]:
# Imports and settings.
from pathlib import Path
import re, gc, logging, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse


SOURCE_FILE = Path(r"F:\BASSFISH_HD\Source_data\1228genes_x_y_axis.h5ad")
OUT_ROOT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene") / "gene_axis_CV_aligned_relative_heatmaps"
OUT_ROOT.mkdir(parents=True, exist_ok=True)


REGION_AXIS_PAIRS = [("d", "x"), ("d", "y"), ("n", "x"), ("n", "y"), ("o", "x")]
TARGET_CELLTYPES = ["D1", "D2"]
TARGET_AXES = sorted({axis for _, axis in REGION_AXIS_PAIRS})
CONDITIONS = ["WT", "Heter", "Homo"]
BATCH_STOP_ON_ERROR = True


REGION_COL = "region_str_manual"
CELLTYPE_COL = "refined_major_cluster"
CONDITION_COL = "condition"
SAMPLE_COL = "sample"
AXIS_COLUMNS = {"x": "region_manual_axis_0_100", "y": "region_manual_axis_y_0_100"}
EXPRESSION_LAYER = "norm_1e4"
DETECTION_LAYER = "counts"

N_BINS = 10
R_THRESHOLD = 0.5
HEATMAP_AGGREGATION = "sample_mean"
PROFILE_NORMALIZATION = "gene_mean1"
ERROR_TYPE = "sem"


USE_WT_GENE_QC = True
MANUAL_GENES = None
QC_MIN_WT_MEAN_NORM = 1.0
QC_MIN_WT_DETECTION = 0.02
QC_MIN_WT_SAMPLE_DETECTION = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.50
QC_MIN_WT_BIN_CELLS = 5
QC_MIN_VALID_WT_BINS = 8


HEATMAP_COLOR_MODE = "relative_mean"
RELATIVE_VMAX = 1.5
ZSCORE_VMAX = 0.5
CV_APPLY_VARIATION_QC = True
CV_MIN_VALID_BINS = 5
CV_MIN_BIN_MEAN_EXPR = 0.02
CV_MIN_EXPRESSED_BINS = 5
CV_MIN_PROFILE_MEAN_EXPR = 0.05
CV_MIN_SAMPLE_FRACTION_PER_CONDITION = 0.50
CV_ABSOLUTE_MAX = 1
CV_DELTA_MAX = 0.5
CV_COLOR_PERCENTILE = 99.0
RELATIVE_RESPECT_CV_QC = True
CMAP = "RdBu_r"
SHOW_GENE_NAMES = False
SHOW_COUNTS = False
PLOT_PER_SAMPLE = True
PLOT_COMBINED = True
SHOW_FIGURES = False
LINE_YLIM = None
DPI = 600
CONDITION_COLORS = {"WT":"#4cd964", "Heter":"#ff9500", "Homo":"#ff3b30"}
HEAT_PANEL_W_MM, HEAT_PANEL_H_MM = 12, 32
HEAT_PANEL_GAP_MM, HEAT_GROUP_GAP_MM = 0.5, 1
LINE_PANEL_W_MM, LINE_PANEL_H_MM = 28, 20

plt.rcParams.update({"font.family":"Arial", "font.size":7, "axes.labelsize":7,
                     "axes.titlesize":7, "xtick.labelsize":7, "ytick.labelsize":7,
                     "legend.fontsize":7, "pdf.fonttype":42, "ps.fonttype":42})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)

assert HEATMAP_AGGREGATION in {"sample_mean","cell_mean"}
assert PROFILE_NORMALIZATION in {"gene_mean1","none"}
assert ERROR_TYPE in {"sem","sd"}
assert HEATMAP_COLOR_MODE in {"relative_mean", "row_zscore"}
assert N_BINS == 10, "This analysis requires ten equal-width bins over 0-100"
print("Input:", SOURCE_FILE)
print("Output:", OUT_ROOT)
TASKS = [(reg, ct, ax) for reg, ax in REGION_AXIS_PAIRS for ct in TARGET_CELLTYPES]
if len(TASKS) != len(set(TASKS)):
    raise ValueError("REGION_AXIS_PAIRS / TARGET_CELLTYPES contains duplicate tasks")
print(f"Batch plan: {len(TASKS)} targets -> {TASKS}")


In [ ]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Data file not found: {SOURCE_FILE}. Check the full filename and path.")
adata = sc.read_h5ad(SOURCE_FILE)
print(adata)
if EXPRESSION_LAYER not in adata.layers:
    raise KeyError(f"Required layers[{EXPRESSION_LAYER!r}] is missing; available layers: {list(adata.layers.keys())}. Fallback to potentially scaled adata.X is disabled.")
for col in [REGION_COL, CELLTYPE_COL, CONDITION_COL, SAMPLE_COL, *[AXIS_COLUMNS[k] for k in TARGET_AXES]]:
    if col not in adata.obs:
        raise KeyError(f"Input obs is missing {col!r}; available columns: {list(adata.obs.columns)}")
if not adata.var_names.is_unique:
    raise ValueError("var_names contains duplicate genes; resolve them first")

X = adata.layers[EXPRESSION_LAYER]
if sparse.issparse(X):

    if len(X.data) and np.nanmin(X.data) < 0:
        raise ValueError("norm_1e4 contains negative values and cannot be used as normalized expression")
else:
    if np.nanmin(X) < 0:
        raise ValueError("norm_1e4 contains negative values and cannot be used as normalized expression")

if DETECTION_LAYER in adata.layers:
    DETECTION_MATRIX = adata.layers[DETECTION_LAYER]
    detection_origin = f"layers[{DETECTION_LAYER}]"
else:
    DETECTION_MATRIX = X
    detection_origin = f"layers[{EXPRESSION_LAYER}] > 0"
    print("Note: no counts layer; gene detection is defined as norm_1e4 > 0")

obs = adata.obs
region_all = obs[REGION_COL].astype(str).to_numpy()
ctype_all = obs[CELLTYPE_COL].astype(str).to_numpy()
condition_all = obs[CONDITION_COL].astype(str).to_numpy()
sample_all = obs[SAMPLE_COL].astype(str).to_numpy()
axis_all = {k: pd.to_numeric(obs[v], errors="coerce").to_numpy(float) for k,v in AXIS_COLUMNS.items() if k in TARGET_AXES}
genes_all = np.asarray(adata.var_names.astype(str))
print("Conditions:", pd.Series(condition_all).value_counts().to_dict())
print("Cell types preview:", sorted(pd.unique(ctype_all))[:35])


In [ ]:
BIN_CENTERS = np.arange(N_BINS) * (100 / N_BINS) + (50 / N_BINS)
BIN_LABELS = [f"{i*10}–{(i+1)*10}" for i in range(N_BINS)]
CLASS_ORDER = ["high_to_low", "non_gradient", "low_to_high"]
CLASS_TITLES = {"high_to_low":"High to low", "non_gradient":"Non-gradient", "low_to_high":"Low to high"}

def safe_nanmean(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a, axis=axis)

def safe_nanstd(a, axis=None, ddof=0):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanstd(a, axis=axis, ddof=ddof)

def mean_rows(mat, rows):
    if not len(rows):
        return np.full(mat.shape[1], np.nan)
    a = mat[rows]
    return np.asarray(a.mean(axis=0)).ravel() if sparse.issparse(a) else safe_nanmean(np.asarray(a), axis=0)

def detect_fraction(mat, rows):
    if not len(rows):
        return np.zeros(mat.shape[1], float)
    a = mat[rows]
    return np.asarray((a > 0).sum(axis=0)).ravel() / len(rows)

def row_zscore(matrix):
    matrix = np.asarray(matrix, float)
    mu = safe_nanmean(matrix, axis=1)[:,None]
    sd = safe_nanstd(matrix, axis=1)[:,None]

    result = np.full(matrix.shape, np.nan, dtype=float)
    valid = np.isfinite(matrix) & np.isfinite(sd) & (sd > 0)
    np.divide(matrix - mu, sd, out=result, where=valid)
    flat = (np.isfinite(sd[:,0]) & (sd[:,0] == 0))
    if flat.any():
        result[flat] = np.where(np.isfinite(matrix[flat]), 0., np.nan)
    return result


def relative_to_own_mean(matrix):
    """Compute bin expression divided by its spatial mean, minus one."""
    matrix = np.asarray(matrix, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        mu = np.nanmean(matrix, axis=1, keepdims=True)
    result = np.full_like(matrix, np.nan, dtype=float)
    np.divide(matrix, mu, out=result,
              where=np.isfinite(matrix) & np.isfinite(mu) & (mu>0))
    result[np.isfinite(result)] -= 1.0
    result[np.isfinite(matrix).sum(axis=1)<CV_MIN_VALID_BINS,:] = np.nan
    return result


def sample_gene_cv(matrix):
    """Compute population SD divided by the positive spatial mean."""
    a=np.asarray(matrix, float)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        mu=np.nanmean(a, axis=1)
        sd=np.nanstd(a, axis=1, ddof=0)
    good=(np.isfinite(a).sum(axis=1)>=CV_MIN_VALID_BINS) & np.isfinite(mu) & (mu>0)
    out=np.full(a.shape[0],np.nan,float)
    out[good]=sd[good]/mu[good]
    return out


def variation_sample_qc_mask(matrix):
    """Apply per-sample expression QC criteria."""
    a=np.asarray(matrix,float)
    valid=np.isfinite(a).sum(axis=1)>=CV_MIN_VALID_BINS
    if not CV_APPLY_VARIATION_QC:
        return valid
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        mu=np.nanmean(a,axis=1)
    expressed=np.sum(np.isfinite(a)&(a>=CV_MIN_BIN_MEAN_EXPR),axis=1)
    return valid & (mu>=CV_MIN_PROFILE_MEAN_EXPR) & (expressed>=CV_MIN_EXPRESSED_BINS)

def wt_pearson_rows(matrix):
    result = np.zeros(matrix.shape[0], float)
    for i,y in enumerate(matrix):
        good = np.isfinite(y)
        if good.sum() >= 3 and np.nanstd(y[good]) > 0:
            result[i] = np.corrcoef(BIN_CENTERS[good], y[good])[0,1]
    return result

def select_gene_mask(target_gene_names, Xsubset, Dsubset, cond, sample, bins, wt_matrix):
    selected = np.ones(len(target_gene_names), bool)
    if MANUAL_GENES is not None:
        wanted = list(dict.fromkeys(str(x) for x in MANUAL_GENES))
        missing = sorted(set(wanted) - set(target_gene_names))
        if missing: raise KeyError(f"Manually selected genes are absent from the panel: {missing}")
        selected &= np.isin(target_gene_names, wanted)
    audit = pd.DataFrame({"gene": target_gene_names, "manual_pass": selected})
    wt_rows = np.flatnonzero(cond == "WT")
    qc = {}
    if USE_WT_GENE_QC:
        if len(wt_rows) == 0: raise ValueError("WT gene QC requires WT cells")
        wt_mean = mean_rows(Xsubset, wt_rows)
        wt_detect = detect_fraction(Dsubset, wt_rows)
        wt_samples = sorted(pd.unique(sample[wt_rows]))
        sf = np.stack([detect_fraction(Dsubset, wt_rows[sample[wt_rows] == s]) for s in wt_samples])
        frac_samples = (sf >= QC_MIN_WT_SAMPLE_DETECTION).mean(axis=0)
        bvalid = np.zeros(len(target_gene_names), int)
        for b in range(N_BINS):
            br = wt_rows[bins[wt_rows] == b]
            if len(br) >= QC_MIN_WT_BIN_CELLS:
                bvalid += (detect_fraction(Dsubset, br) > 0).astype(int)
        qc_pass = ((wt_mean >= QC_MIN_WT_MEAN_NORM) &
                   (wt_detect >= QC_MIN_WT_DETECTION) &
                   (frac_samples >= QC_MIN_WT_SAMPLE_FRACTION) &
                   (bvalid >= QC_MIN_VALID_WT_BINS))
        selected &= qc_pass
        qc = {"WT_mean_norm_1e4":wt_mean, "WT_detection_fraction":wt_detect,
              "WT_fraction_samples_ge_1pct":frac_samples,
              "WT_valid_expressed_bins":bvalid, "WT_QC_pass":qc_pass}
        for k,v in qc.items(): audit[k] = v
    audit["selected"] = selected
    if not selected.any(): raise ValueError("No genes passed filtering; check the criteria")
    return selected, audit


In [ ]:
def plot_heat_panels(panels, sorted_names, counts_by_key, out_pdf, axis_label, page_title):

    if not panels: return
    left, right, bottom, top, cb_gap, cb_w = 16, 5, 17, 16, 4, 3
    xs = []; cursor = left; prev_group = None
    for title, group, _ in panels:
        if prev_group is not None and group != prev_group:
            cursor += HEAT_GROUP_GAP_MM - HEAT_PANEL_GAP_MM
        xs.append(cursor)
        cursor += HEAT_PANEL_W_MM + HEAT_PANEL_GAP_MM
        prev_group = group
    heat_right = xs[-1] + HEAT_PANEL_W_MM
    fw = heat_right + cb_gap + cb_w + right
    fh = bottom + HEAT_PANEL_H_MM + top
    fig = plt.figure(figsize=(fw/25.4, fh/25.4))
    try:

        fig.text(left/fw, (bottom + HEAT_PANEL_H_MM + 11)/fh, page_title,
                 ha="left", va="center", fontsize=8)
        im = None
        for i, ((title, group, z), xp) in enumerate(zip(panels, xs)):
            ax = fig.add_axes([xp/fw, bottom/fh, HEAT_PANEL_W_MM/fw, HEAT_PANEL_H_MM/fh])
            im = ax.imshow(np.ma.masked_invalid(z), cmap=CMAP, vmin=-ZSCORE_VMAX,
                           vmax=ZSCORE_VMAX, aspect="auto", interpolation="nearest",
                           rasterized=False)
            ax.set_title(title, pad=2)
            count = counts_by_key[(title,group)]
            labels = ([f"{b+1}\n{int(count[b])}" for b in range(N_BINS)] if SHOW_COUNTS
                      else [str(b+1) for b in range(N_BINS)])
            ax.set_xticks(np.arange(N_BINS)); ax.set_xticklabels(labels, fontsize=6)
            ax.set_xlabel(axis_label, fontsize=7)
            if i == 0 and SHOW_GENE_NAMES and len(sorted_names) <= 65:
                ax.set_yticks(np.arange(len(sorted_names)))
                ax.set_yticklabels(sorted_names, fontsize=5)
            else:
                ax.set_yticks([])
            for s in ax.spines.values(): s.set_linewidth(.5)
            ax.tick_params(length=2, width=.5, pad=1)
        if im is not None:
            cax = fig.add_axes([(heat_right+cb_gap)/fw, bottom/fh, cb_w/fw, HEAT_PANEL_H_MM/fh])
            cb = fig.colorbar(im,cax=cax)
            cb.set_label("Row z-score")
            cb.ax.tick_params(width=.5, length=2, pad=1)
            cb.outline.set_linewidth(.5)
        fig.savefig(out_pdf, dpi=DPI, bbox_inches="tight", pad_inches=.03, facecolor="white")
        if SHOW_FIGURES: plt.show()
    finally:
        plt.close(fig)
    print("Saved",out_pdf)

def plot_class_profiles(summary, category_counts, out_pdf, axis_label, page_title):
    left, right, bottom, top, gap = 13, 6, 14, 15, 7
    fw = left + 3*LINE_PANEL_W_MM + 2*gap + right
    fh = bottom + LINE_PANEL_H_MM + top
    fig = plt.figure(figsize=(fw/25.4, fh/25.4))
    try:
        fig.text(left/fw, (bottom + LINE_PANEL_H_MM + 10)/fh, page_title,
                 ha="left", va="center", fontsize=8)
        axes=[]
        for j,cat in enumerate(CLASS_ORDER):
            xp=left + j*(LINE_PANEL_W_MM + gap)
            ax=fig.add_axes([xp/fw, bottom/fh, LINE_PANEL_W_MM/fw, LINE_PANEL_H_MM/fh])
            axes.append(ax)
            for cond in CONDITIONS:
                sub = summary[(summary["category"]==cat)&(summary["condition"]==cond)].sort_values("bin")
                if sub.empty: continue
                x = sub["bin_center"].to_numpy(float)
                y = sub["mean"].to_numpy(float)
                e = sub[ERROR_TYPE].to_numpy(float)
                ax.plot(x,y,lw=1.0,label=cond,color=CONDITION_COLORS[cond])
                g=np.isfinite(e) & np.isfinite(y)
                if g.any():
                    ax.fill_between(x,y-e,y+e,where=g,color=CONDITION_COLORS[cond],alpha=.18,lw=0)
            if PROFILE_NORMALIZATION == "gene_mean1":
                ax.axhline(1,color="0.7",lw=.5,ls="--")
            ax.set_xlim(0,100);ax.set_xticks([0,20,40,60,80,100]);ax.set_xlabel(axis_label)
            if LINE_YLIM is not None: ax.set_ylim(*LINE_YLIM)
            ax.set_title(f"{CLASS_TITLES[cat]} (n={category_counts.get(cat,0)})",pad=2)
            for s in ax.spines.values():s.set_linewidth(.5)
            ax.tick_params(width=.5,length=2,pad=1)
        axes[0].set_ylabel("Relative expression (gene mean=1)" if PROFILE_NORMALIZATION == "gene_mean1"
                           else "Mean expression (norm_1e4)")
        axes[-1].legend(frameon=False,fontsize=6,handlelength=1.4,loc="best")
        fig.savefig(out_pdf,dpi=DPI,bbox_inches="tight",pad_inches=.03,facecolor="white")
        if SHOW_FIGURES:plt.show()
    finally:
        plt.close(fig)
    print("Saved",out_pdf)


In [ ]:
def cv_heatmap_plot(prefix, names, cv_by_condition, delta_by_condition, outpath):

    n=len(names)
    absolute=np.column_stack([cv_by_condition[c] for c in CONDITIONS])
    diffs=np.column_stack([delta_by_condition['Heter'],delta_by_condition['Homo']])
    finite_cv=absolute[np.isfinite(absolute)]
    finite_d=diffs[np.isfinite(diffs)]
    vmax=(CV_ABSOLUTE_MAX if CV_ABSOLUTE_MAX is not None else
          max(.1, float(np.percentile(finite_cv,CV_COLOR_PERCENTILE))) if len(finite_cv) else 1.)
    dmax=(CV_DELTA_MAX if CV_DELTA_MAX is not None else
          max(.05, float(np.percentile(np.abs(finite_d),CV_COLOR_PERCENTILE))) if len(finite_d) else .5)

    fig=plt.figure(figsize=(4.35, max(2.1,min(8., n*.015+1.2))))
    try:
        ax1=fig.add_axes([.10,.11,.34,.79]); ax2=fig.add_axes([.56,.11,.22,.79])
        cmap1=plt.get_cmap('Reds').copy();cmap1.set_bad('white')
        cmap2=plt.get_cmap('RdBu_r').copy();cmap2.set_bad('white')
        im1=ax1.imshow(np.ma.masked_invalid(absolute),aspect='auto',interpolation='nearest',
                       cmap=cmap1,vmin=0,vmax=vmax,rasterized=False)
        im2=ax2.imshow(np.ma.masked_invalid(diffs),aspect='auto',interpolation='nearest',
                       cmap=cmap2,vmin=-dmax,vmax=dmax,rasterized=False)
        ax1.set_xticks(np.arange(3),CONDITIONS);ax2.set_xticks([0,1],['Heter−WT','Homo−WT'])
        for ax in (ax1,ax2):
            ax.set_yticks([]);ax.tick_params(axis='x',length=0,labelrotation=30,labelsize=6)
            for sp in ax.spines.values():sp.set_linewidth(.4)
        if SHOW_GENE_NAMES and n<=65:
            ax1.set_yticks(range(n));ax1.set_yticklabels(names,fontsize=4.5)
        fig.text(.10,.955,f'{prefix.replace("_"," | ")}  |  per-gene CV (animal mean) and ΔCV',fontsize=8,va='center')
        ca1=fig.add_axes([.455,.11,.022,.79]);ca2=fig.add_axes([.79,.11,.022,.79])
        cb1=fig.colorbar(im1,cax=ca1);cb1.set_label('CV',fontsize=7)
        cb2=fig.colorbar(im2,cax=ca2);cb2.set_label('ΔCV',fontsize=7)
        cb1.ax.tick_params(labelsize=6,length=2);cb2.ax.tick_params(labelsize=6,length=2)
        fig.savefig(outpath,dpi=DPI,bbox_inches='tight',pad_inches=.03,facecolor='white')
    finally:
        plt.close(fig)
    print('Saved',outpath)


def cv_by_animal_for_target(prefix,samples_raw,meta,sample_info,outdir):

    genes=meta['gene'].to_numpy(str)
    records=[]
    cv_array={}
    pass_array={}
    for c,s in sample_info:
        mat=samples_raw[(c,s)]
        val=sample_gene_cv(mat)
        passes=variation_sample_qc_mask(mat)
        val[~passes]=np.nan
        cv_array[(c,s)]=val
        pass_array[(c,s)]=passes
        for i,g in enumerate(genes):
            records.append({'condition':c,'sample':s,'gene':g,'cv':val[i],
                            'pass_variation_sample_QC':bool(passes[i]),
                            'n_valid_bins':int(np.isfinite(mat[i]).sum()),
                            'profile_mean':float(safe_nanmean(mat[i]))})
    df=pd.DataFrame(records)
    df.to_csv(outdir/f'{prefix}_12_gene_sample_CV_matched_variation.csv',index=False)
    cv_by_c={}; count_by_c={}; fraction_by_c={}
    crows=[]
    for c in CONDITIONS:
        keys=[k for k in sample_info if k[0]==c]
        values=np.stack([cv_array[k] for k in keys],axis=0)
        nfinite=np.isfinite(values).sum(axis=0)

        req=(max(1,int(np.ceil(len(keys)*CV_MIN_SAMPLE_FRACTION_PER_CONDITION)))
             if CV_APPLY_VARIATION_QC else 1)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            avg=np.nanmean(values,axis=0)
            sd=np.nanstd(values,axis=0,ddof=1) if len(keys)>1 else np.full(values.shape[1],np.nan)
        avg[nfinite<req]=np.nan
        cv_by_c[c]=avg;count_by_c[c]=nfinite
        fraction_by_c[c]=nfinite/len(keys)
        for i,g in enumerate(genes):
            crows.append({'gene':g,'condition':c,'cv_animal_mean':avg[i],
                          'n_animals_valid':nfinite[i], 'n_animals_total':len(keys),
                          'min_animals_required':req,'cv_sd_animals':sd[i],
                          'cv_sem_animals':sd[i]/np.sqrt(nfinite[i]) if nfinite[i]>1 else np.nan})
    paired=np.isfinite(cv_by_c['WT']) & np.isfinite(cv_by_c['Heter']) & np.isfinite(cv_by_c['Homo'])
    differences={'Heter':np.where(paired,cv_by_c['Heter']-cv_by_c['WT'],np.nan),
                 'Homo':np.where(paired,cv_by_c['Homo']-cv_by_c['WT'],np.nan)}
    pd.DataFrame(crows).to_csv(outdir/f'{prefix}_13_gene_condition_CV_matched_variation.csv',index=False)
    rows=[]
    for i,g in enumerate(genes):
        rows.append({'gene':g,'spatial_category':meta['spatial_category'].iat[i],
                     'WT_CV':cv_by_c['WT'][i], 'Heter_CV':cv_by_c['Heter'][i],
                     'Homo_CV':cv_by_c['Homo'][i],
                     'Heter_minus_WT':differences['Heter'][i],
                     'Homo_minus_WT':differences['Homo'][i],
                     'paired_all_conditions':bool(paired[i])})
    pd.DataFrame(rows).to_csv(outdir/f'{prefix}_14_gene_CV_signed_delta_WT.csv',index=False)
    if np.any(paired):
        print(prefix,'CV paired genes:',int(paired.sum()),
              'median WT/Heter/Homo:',*[round(float(np.nanmedian(cv_by_c[c][paired])),4) for c in CONDITIONS])
    else:
        print(prefix,'WARNING: no paired genes passed CV sample/condition QC')
    return cv_by_c,differences


In [ ]:
def analyze_one(region_target, celltype_target, axis_name):
    prefix=f"{region_target}_{celltype_target}_{axis_name}"
    od = OUT_ROOT
    page_prefix = f"{celltype_target} | {region_target}-{axis_name} axis"
    axis_values=axis_all[axis_name]
    pattern=rf"(?:^|_){re.escape(celltype_target)}(?:$|_)"
    ct_match=pd.Series(ctype_all).str.contains(pattern,regex=True,na=False).to_numpy(bool)
    valid = (region_all == region_target) & ct_match & np.isfinite(axis_values)
    valid &= (axis_values >= 0)&(axis_values <=100)& np.isin(condition_all,CONDITIONS)
    valid &= (sample_all != "nan") & (sample_all != "<NA>") & (sample_all != "")
    idx=np.flatnonzero(valid)
    if len(idx)==0:
        raise ValueError(f"{prefix}: no matching cells. Check region, celltype, and axis: {sorted(pd.unique(ctype_all))[:40]}")
    co=condition_all[idx]; sa=sample_all[idx]; a=axis_values[idx]
    bi=np.minimum(np.floor(a / (100/N_BINS)).astype(int),N_BINS-1)

    Xsub=X[idx]
    Dsub=DETECTION_MATRIX[idx] if USE_WT_GENE_QC else None
    sample_data={}; sample_counts={}; sample_info=[]; bin_qc=[]
    for c in CONDITIONS:
        ss=sorted(pd.unique(sa[co==c]))
        if not ss: raise ValueError(f"{prefix}: group {c} has no cells; the complete three-condition plot cannot be drawn")
        for s in ss:
            si=np.flatnonzero((co==c)&(sa==s))
            profile=np.full((adata.n_vars,N_BINS),np.nan)
            count=np.zeros(N_BINS,dtype=int)
            for b in range(N_BINS):
                ri=si[bi[si]==b]
                count[b]=len(ri)
                if len(ri): profile[:,b]=mean_rows(Xsub,ri)
                bin_qc.append({"region":region_target,"celltype":celltype_target,
                               "axis":axis_name,"condition":c,"sample":s,"bin":b+1,
                               "axis_left":b*10,"axis_right":(b+1)*10,"n_cells":len(ri)})
            sample_info.append((c,s))
            sample_data[(c,s)] = profile
            sample_counts[(c,s)] = count
    pooled_data={}; pooled_counts={}
    for c in CONDITIONS:
        keys=[(cc,s) for cc,s in sample_info if cc==c]
        if HEATMAP_AGGREGATION=="sample_mean":
            pooled_data[c] = safe_nanmean(np.stack([sample_data[k] for k in keys]),axis=0)
        else:
            pooled_data[c] = np.full((adata.n_vars,N_BINS),np.nan)
            for b in range(N_BINS):
                rows=np.flatnonzero((co==c)&(bi==b))
                if len(rows): pooled_data[c][:,b] = mean_rows(Xsub,rows)
        pooled_counts[c]=np.sum([sample_counts[k] for k in keys],axis=0)

    selected,gene_qc = select_gene_mask(genes_all, Xsub, Dsub, co, sa, bi, pooled_data["WT"])
    gene_qc.to_csv(od / f"{prefix}_00_gene_filter_QC.csv",index=False)
    selected_genes=genes_all[selected]
    nsel=len(selected_genes)
    wt_r=wt_pearson_rows(pooled_data["WT"][selected])
    cats=np.full(nsel,"non_gradient",dtype=object)
    cats[wt_r <= -R_THRESHOLD]="high_to_low"
    cats[wt_r >= R_THRESHOLD]="low_to_high"
    meta=pd.DataFrame({"gene":selected_genes,"WT_axis_r":wt_r,"spatial_category":cats,
                       "pattern_strength":np.abs(wt_r)})
    meta=meta.sort_values(["WT_axis_r","gene"],ascending=[True,True]).reset_index(drop=True)
    order=pd.Index(selected_genes).get_indexer(meta["gene"])
    meta.insert(0,"rank",np.arange(1,nsel+1))
    meta.to_csv(od / f"{prefix}_01_WT_fixed_gene_order_and_classes.csv",index=False)

    pooled_raw={c:pooled_data[c][selected][order] for c in CONDITIONS}
    pooled_z={c:row_zscore(pooled_raw[c]) for c in CONDITIONS}
    samples_raw={(c,s):sample_data[(c,s)][selected][order] for c,s in sample_info}
    samples_z={(c,s):row_zscore(samples_raw[(c,s)]) for c,s in sample_info}

    samples_rel={}
    for c,s in sample_info:
        arr=relative_to_own_mean(samples_raw[(c,s)])
        if RELATIVE_RESPECT_CV_QC:
            arr[~variation_sample_qc_mask(samples_raw[(c,s)]),:]=np.nan
        samples_rel[(c,s)]=arr
    pooled_rel={}
    for c in CONDITIONS:
        subset=[samples_rel[(cc,ss)] for cc,ss in sample_info if cc==c]

        pooled_rel[c]=safe_nanmean(np.stack(subset),axis=0)
        if RELATIVE_RESPECT_CV_QC:
            ok=np.sum(np.isfinite(np.stack(subset)).any(axis=2),axis=0)
            minimum=(max(1,int(np.ceil(len(subset)*CV_MIN_SAMPLE_FRACTION_PER_CONDITION)))
                     if CV_APPLY_VARIATION_QC else 1)
            pooled_rel[c][ok<minimum,:]=np.nan
    if HEATMAP_COLOR_MODE == 'relative_mean':
        plot_pooled,plot_samples=pooled_rel,samples_rel
    else:
        plot_pooled,plot_samples=pooled_z,samples_z

    cv_condition,cv_delta=cv_by_animal_for_target(prefix,samples_raw,meta,sample_info,od)
    cv_heatmap_plot(prefix,meta['gene'].to_list(),cv_condition,cv_delta,
                    od/f'{prefix}_15_CV_gene_condition_and_signed_change_heatmap.pdf')

    record_raw=[];record_z=[]
    for c in CONDITIONS:
        for i,g in enumerate(meta["gene"]):
            rec={"condition":c,"gene":g}
            record_raw.append({**rec,**{f"bin_{b+1}":pooled_raw[c][i,b] for b in range(N_BINS)}})
            record_z.append({**rec,**{f"bin_{b+1}":pooled_z[c][i,b] for b in range(N_BINS)}})
    pd.DataFrame(record_raw).to_csv(od / f"{prefix}_02_condition_bin_mean_expression.csv",index=False)
    pd.DataFrame(record_z).to_csv(od / f"{prefix}_03_condition_bin_row_zscore.csv",index=False)

    rel_condition=[]; rel_sample=[]
    for c in CONDITIONS:
        for i,g in enumerate(meta['gene']):
            rel_condition.append({'condition':c,'gene':g,
                                  **{f'bin_{b+1}':pooled_rel[c][i,b] for b in range(N_BINS)}})
    for c,ss in sample_info:
        for i,g in enumerate(meta['gene']):
            rel_sample.append({'condition':c,'sample':ss,'gene':g,
                               **{f'bin_{b+1}':samples_rel[(c,ss)][i,b] for b in range(N_BINS)}})
    pd.DataFrame(rel_condition).to_csv(od/f'{prefix}_16_condition_relative_to_sample_mean.csv',index=False)
    pd.DataFrame(rel_sample).to_csv(od/f'{prefix}_17_sample_relative_to_own_mean.csv',index=False)


    record_samples=[]
    for c,s in sample_info:
        for i,g in enumerate(meta["gene"]):
            record_samples.append({"condition":c,"sample":s,"gene":g,
                                   **{f"bin_{b+1}":samples_raw[(c,s)][i,b] for b in range(N_BINS)}})
    pd.DataFrame(record_samples).to_csv(od / f"{prefix}_04_sample_gene_bin_mean_expression.csv",index=False)
    pd.DataFrame(bin_qc).to_csv(od / f"{prefix}_05_sample_bin_cell_counts.csv",index=False)


    class_records=[]
    classes=meta["spatial_category"].to_numpy()
    for c,s in sample_info:
        mat=samples_raw[(c,s)]
        if PROFILE_NORMALIZATION=="gene_mean1":
            gene_mean=safe_nanmean(mat,axis=1)[:,None]
            norm=np.full_like(mat,np.nan,dtype=float)
            np.divide(mat,gene_mean,out=norm,where=np.isfinite(mat)&np.isfinite(gene_mean)&(gene_mean>0))
        else:
            norm=mat.copy()
        for cat in CLASS_ORDER:
            gm=classes==cat
            if not gm.any(): continue
            m=safe_nanmean(norm[gm],axis=0)
            for b,v in enumerate(m):
                class_records.append({"condition":c,"sample":s,"category":cat,"bin":b+1,
                                      "bin_center":BIN_CENTERS[b],"value":v,
                                      "n_class_genes":int(gm.sum())})
    class_df=pd.DataFrame(class_records)
    class_df.to_csv(od / f"{prefix}_06_sample_class_profiles.csv",index=False)
    class_summary=class_df.groupby(["condition","category","bin","bin_center"],observed=True)["value"].agg(["mean","std","count"]).reset_index()
    class_summary["sem"] = class_summary["std"] / np.sqrt(class_summary["count"])
    class_summary.to_csv(od / f"{prefix}_07_condition_class_mean_sem.csv",index=False)


    names=meta["gene"].tolist()
    cond_panels=[(c,c,plot_pooled[c]) for c in CONDITIONS]
    cond_map={(c,c):pooled_counts[c] for c in CONDITIONS}
    sample_panels=[(f"{s}\n{c}",c,plot_samples[(c,s)]) for c,s in sample_info]
    sample_map={(f"{s}\n{c}",c):sample_counts[(c,s)] for c,s in sample_info}
    axis_title=f"{axis_name} axis (0–100)"
    plot_heat_panels(cond_panels,names,cond_map,od / f"{prefix}_08_WT_Heter_Homo_heatmap.pdf",axis_title,
                     f"{page_prefix} | WT / Heter / Homo pooled heatmap")
    if PLOT_PER_SAMPLE:
        plot_heat_panels(sample_panels,names,sample_map,od / f"{prefix}_09_all_samples_heatmap.pdf",axis_title,
                         f"{page_prefix} | per-sample heatmap")
    if PLOT_COMBINED:
        group_panels=[(f"Pooled {c}","Pooled",plot_pooled[c]) for c in CONDITIONS]
        group_map={(f"Pooled {c}","Pooled"):pooled_counts[c] for c in CONDITIONS}
        plot_heat_panels(group_panels+sample_panels,names,{**group_map,**sample_map},
                         od / f"{prefix}_10_condition_plus_samples_heatmap.pdf",axis_title,
                         f"{page_prefix} | pooled conditions + per-sample heatmap")
    category_counts=meta["spatial_category"].value_counts().to_dict()
    plot_class_profiles(class_summary,category_counts,od / f"{prefix}_11_WT_gene_class_profiles_mean_sem.pdf",axis_title,
                        f"{page_prefix} | WT-defined gene classes | mean ± {ERROR_TYPE.upper()}")
    print(prefix,"n_cells",len(idx),"n_genes",nsel,"categories",category_counts)
    del Xsub, Dsub, sample_data, samples_raw, samples_z, samples_rel, pooled_rel
    gc.collect()
    return {"region":region_target,"celltype":celltype_target,"axis":axis_name,"n_cells":len(idx),
            "n_genes":nsel,"n_WT_samples":len(set(sa[co=="WT"])),
            "n_Heter_samples":len(set(sa[co=="Heter"])),"n_Homo_samples":len(set(sa[co=="Homo"])),
            "expression_source":EXPRESSION_LAYER,"QC_enabled":USE_WT_GENE_QC,
            "output":str(od)}


In [ ]:
results=[]
for job_idx, (region, ct, axis) in enumerate(TASKS, 1):
    print(f"\n{'='*65}\n[{job_idx}/{len(TASKS)}] {region} / {ct} / {axis}")
    try:
        info = analyze_one(region, ct, axis)
        results.append({**info, "status":"saved", "error":""})
    except Exception as exc:
        import traceback
        print(f"ERROR [{region}_{ct}_{axis}]: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        results.append({"region":region,"celltype":ct,"axis":axis,
                        "status":"error","error":f"{type(exc).__name__}: {exc}"})
        if BATCH_STOP_ON_ERROR:
            pd.DataFrame(results).to_csv(OUT_ROOT/"_analysis_summary.csv",index=False)
            raise

    pd.DataFrame(results).to_csv(OUT_ROOT/"_analysis_summary.csv",index=False)
summary = pd.DataFrame(results)
print("\nCompleted:", OUT_ROOT)
print(summary[["region","celltype","axis","status"]].to_string(index=False))
print(f"Saved {int((summary['status']=='saved').sum())}/{len(TASKS)} combinations.")


## Panels D, F, J, L, P | WT-defined category profiles

Uses per-animal gene-mean normalization, WT-defined classes, and animal mean/SEM. The batch also exports D2 results.


In [ ]:
# Imports and settings.
from pathlib import Path
import re, gc, logging, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from matplotlib.lines import Line2D


SOURCE_FILE = Path(r"F:\BASSFISH_HD\Source_data\1228genes_x_y_axis.h5ad")
OUT_ROOT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene") / "gene_gradient_5axes_WTclasses_optional_QC"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

AXIS_SPECS = [("d", "x"), ("d", "y"), ("n", "x"), ("n", "y"), ("o", "x")]
TARGET_CELLTYPES = ["D1", "D2"]
CONDITIONS = ["WT", "Heter", "Homo"]

REGION_COL = "region_str_manual"
CELLTYPE_COL = "refined_major_cluster"
CONDITION_COL = "condition"
SAMPLE_COL = "sample"
AXIS_COLS = {"x": "region_manual_axis_0_100", "y": "region_manual_axis_y_0_100"}
EXPRESSION_LAYER = "norm_1e4"
DETECTION_LAYER = "counts"


N_BINS = 10
R_THRESHOLD = 0.5
PROFILE_NORMALIZATION = "gene_mean1"
ERROR_TYPE = "sem"
CONDITION_COLORS = {"WT":"#4cd964", "Heter":"#ff9500", "Homo":"#ff3b30"}
CLASS_ORDER = ["high_to_low", "non_gradient", "low_to_high"]
CLASS_LINESTYLE = {
    "high_to_low": "-",
    "non_gradient": ":",
    "low_to_high": "--"
}
CLASS_LABEL = {
    "high_to_low": "High to low",
    "non_gradient": "Non-gradient",
    "low_to_high": "Low to high"
}


USE_WT_GENE_QC = True
MANUAL_GENES = None
QC_MIN_WT_MEAN_NORM = 1.0
QC_MIN_WT_DETECTION = 0.02
QC_MIN_WT_SAMPLE_DETECTION = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.50
QC_MIN_WT_BIN_CELLS = 5
QC_MIN_VALID_WT_BINS = 8


Y_LIMITS = (0.5, 1.5)
LINEWIDTH = 0.9
ERROR_ALPHA = 0.10
DPI = 600
FONT_PT = 6
PANEL_W_MM, PANEL_H_MM, PANEL_GAP_MM = 40, 28, 20
LEFT_MM, RIGHT_MM, BOTTOM_MM, TOP_MM = 10, 5, 10, 8
SHOW_FIGURES = False

assert N_BINS == 10, "This analysis requires 10 equal-width bins"
assert PROFILE_NORMALIZATION in ("gene_mean1", "none")
assert ERROR_TYPE in ("sem", "sd")
assert 0 < R_THRESHOLD <= 1
assert len(set(AXIS_SPECS)) == len(AXIS_SPECS)
plt.rcParams.update({
    "font.family": "Arial", "font.size": FONT_PT,
    "axes.labelsize": FONT_PT, "axes.titlesize": FONT_PT,
    "xtick.labelsize": FONT_PT, "ytick.labelsize": FONT_PT,
    "legend.fontsize": FONT_PT, "pdf.fonttype": 42, "ps.fonttype": 42,
})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)
print("Source:", SOURCE_FILE)
print("Output:", OUT_ROOT)


In [ ]:
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"Source data not found: {SOURCE_FILE}")
adata = sc.read_h5ad(SOURCE_FILE)
print(adata)
needed = [REGION_COL, CELLTYPE_COL, CONDITION_COL, SAMPLE_COL] + [AXIS_COLS[a] for _, a in AXIS_SPECS]
missing = sorted(set(needed) - set(adata.obs.columns))
if missing:
    raise KeyError(f"obs is missing required columns: {missing}")
if EXPRESSION_LAYER not in adata.layers:
    raise KeyError(f"Missing layers[{EXPRESSION_LAYER!r}]; available layers: {list(adata.layers.keys())}. Fallback to adata.X is disabled")
if not adata.var_names.is_unique:
    raise ValueError("Duplicate gene names; resolve them first")
X = adata.layers[EXPRESSION_LAYER]
DETECTION = adata.layers[DETECTION_LAYER] if DETECTION_LAYER in adata.layers else X
DETECTION_ORIGIN = DETECTION_LAYER if DETECTION_LAYER in adata.layers else f"{EXPRESSION_LAYER}>0"
print("Expression:", EXPRESSION_LAYER, "Detection:", DETECTION_ORIGIN)

obs = adata.obs
condition_all = obs[CONDITION_COL].astype("string").fillna("").astype(str).to_numpy()
sample_all = obs[SAMPLE_COL].astype("string").fillna("").astype(str).to_numpy()
region_all = obs[REGION_COL].astype("string").fillna("").astype(str).to_numpy()
ctype_all = obs[CELLTYPE_COL].astype("string").fillna("").astype(str).to_numpy()
axis_values = {
    a: pd.to_numeric(obs[AXIS_COLS[a]], errors="coerce").to_numpy(float)
    for a in sorted({axis for _, axis in AXIS_SPECS})
}
genes_all = np.asarray(adata.var_names.astype(str))
BIN_CENTERS = np.arange(N_BINS) * 10 + 5
print("QC enabled:", USE_WT_GENE_QC, "Manual genes:", MANUAL_GENES)


In [ ]:
def safe_nanmean(values, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(values, axis=axis)

def mean_rows(mat, idx):
    if len(idx) == 0:
        return np.full(mat.shape[1], np.nan)
    z = mat[idx]
    return np.asarray(z.mean(axis=0)).ravel() if sparse.issparse(z) else safe_nanmean(np.asarray(z), axis=0)

def detection_fraction(mat, idx):
    if len(idx) == 0:
        return np.zeros(mat.shape[1], float)
    z = mat[idx]
    if sparse.issparse(z):
        return np.asarray((z > 0).sum(axis=0)).ravel() / len(idx)
    return np.mean(np.asarray(z) > 0, axis=0)

def pearson_rows(matrix, x):
    out = np.full(matrix.shape[0], np.nan)
    for i, row in enumerate(matrix):
        good = np.isfinite(row)
        if good.sum() < 3 or np.nanstd(row[good]) == 0:
            out[i] = 0.0
        else:
            out[i] = np.corrcoef(x[good], row[good])[0, 1]
    return out

def gene_filter(mask_global, bin_index, sample_local, cond_local, sample_matrices, prefix):
    """Select genes and export the QC audit."""
    gene_selected = np.ones(adata.n_vars, dtype=bool)
    manual_selected = np.ones(adata.n_vars, dtype=bool)
    if MANUAL_GENES is not None:
        requested = [str(g) for g in MANUAL_GENES]
        missing_genes = sorted(set(requested) - set(genes_all))
        if missing_genes:
            raise KeyError(f"{prefix}: manually selected genes not found: {missing_genes}")
        manual_selected = np.isin(genes_all, requested)
        gene_selected &= manual_selected
    audit = pd.DataFrame({"gene": genes_all, "manual_pass": manual_selected})
    if USE_WT_GENE_QC:
        wt_local = np.flatnonzero(cond_local == "WT")
        if len(wt_local) == 0:
            raise ValueError(f"{prefix}: no WT cells for WT QC")

        global_ids = np.flatnonzero(mask_global)
        wt_global = global_ids[wt_local]
        wt_mean = mean_rows(X, wt_global)
        wt_detect = detection_fraction(DETECTION, wt_global)
        wt_samples = sorted(pd.unique(sample_local[wt_local]))
        wt_sample_det = np.stack([
            detection_fraction(DETECTION, wt_global[sample_local[wt_local] == s])
            for s in wt_samples
        ])
        animal_fraction = np.mean(wt_sample_det >= QC_MIN_WT_SAMPLE_DETECTION, axis=0)
        expressed_bins = np.zeros(adata.n_vars, dtype=int)
        for b in range(N_BINS):
            w_bin_global = wt_global[bin_index[wt_local] == b]
            if len(w_bin_global) >= QC_MIN_WT_BIN_CELLS:
                expressed_bins += (detection_fraction(DETECTION, w_bin_global) > 0).astype(int)
        wt_pass = ((wt_mean >= QC_MIN_WT_MEAN_NORM) &
                   (wt_detect >= QC_MIN_WT_DETECTION) &
                   (animal_fraction >= QC_MIN_WT_SAMPLE_FRACTION) &
                   (expressed_bins >= QC_MIN_VALID_WT_BINS))
        gene_selected &= wt_pass
        audit["WT_mean_norm_1e4"] = wt_mean
        audit["WT_detection_fraction"] = wt_detect
        audit["WT_fraction_samples_detect_ge_1pct"] = animal_fraction
        audit["WT_valid_expressed_bins"] = expressed_bins
        audit["WT_expression_QC_pass"] = wt_pass
    audit["selected"] = gene_selected
    if not np.any(gene_selected):
        raise ValueError(f"{prefix}: no genes passed filtering; check USE_WT_GENE_QC and its thresholds")
    return gene_selected, audit


In [ ]:
def analyze_axis(target_celltype, target_region, target_axis):
    prefix = f"{target_celltype}_{target_region}-{target_axis}"
    axis = axis_values[target_axis]
    pattern = rf"(?:^|_){re.escape(target_celltype)}(?:$|_)"
    type_matched = pd.Series(ctype_all).str.contains(pattern, regex=True, na=False).to_numpy(bool)
    mask = (region_all == target_region) & type_matched
    mask &= np.isfinite(axis) & (axis >= 0) & (axis <= 100)
    mask &= np.isin(condition_all, CONDITIONS) & (sample_all != "")
    chosen = np.flatnonzero(mask)
    if len(chosen) == 0:
        raise ValueError(f"{prefix}: no matching cells; check region/celltype/axis")
    cond = condition_all[chosen]
    sample = sample_all[chosen]
    bins = np.minimum(np.floor(axis[chosen] / (100 / N_BINS)).astype(int), N_BINS - 1)


    sample_profiles = {}
    count_records = []
    sample_names = {}
    for c in CONDITIONS:
        samples = sorted(pd.unique(sample[cond == c]))
        sample_names[c] = samples
        if c == "WT" and not samples:
            raise ValueError(f"{prefix}: no WT animals to define reference classes")
        if not samples:
            warnings.warn(f"{prefix}: no animals for {c}; this group will be absent from outputs", RuntimeWarning)
        for s in samples:
            rows = np.flatnonzero((cond == c) & (sample == s))
            profile = np.full((adata.n_vars, N_BINS), np.nan, dtype=float)
            for b in range(N_BINS):
                sb = rows[bins[rows] == b]
                if len(sb):
                    profile[:, b] = mean_rows(X, chosen[sb])
                count_records.append({"celltype":target_celltype,"region":target_region,"axis":target_axis,
                                      "condition":c,"sample":s,"bin":b+1,"bin_center":BIN_CENTERS[b],
                                      "n_cells":len(sb)})
            sample_profiles[(c,s)] = profile

    wt_mats = [mat for (c,s), mat in sample_profiles.items() if c == "WT"]
    wt_mean = safe_nanmean(np.stack(wt_mats), axis=0)
    wt_r_all = pearson_rows(wt_mean, BIN_CENTERS)
    selected, audit = gene_filter(mask, bins, sample, cond, sample_profiles, prefix)
    gene_class_all = np.full(adata.n_vars, "non_gradient", dtype=object)
    gene_class_all[wt_r_all <= -R_THRESHOLD] = "high_to_low"
    gene_class_all[wt_r_all >= R_THRESHOLD] = "low_to_high"
    audit["WT_axis_r"] = wt_r_all
    audit["WT_category_if_selected"] = np.where(selected, gene_class_all, "filtered")
    audit["celltype"] = target_celltype
    audit["region"] = target_region
    audit["axis"] = target_axis
    selected_meta = audit.loc[selected, ["gene", "WT_axis_r", "WT_category_if_selected"]].copy()
    selected_meta = selected_meta.rename(columns={"WT_category_if_selected":"spatial_category"})
    selected_indices = np.flatnonzero(selected)

    norm_profiles = {}
    for key, mat in sample_profiles.items():
        chosen_mat = mat[selected_indices]
        if PROFILE_NORMALIZATION == "gene_mean1":
            gmean = safe_nanmean(chosen_mat, axis=1)[:,None]
            norm = np.divide(chosen_mat, gmean, out=np.full_like(chosen_mat, np.nan),
                             where=np.isfinite(gmean) & (gmean > 0))
        else:
            norm = chosen_mat.copy()
        norm_profiles[key] = norm

    cats = selected_meta["spatial_category"].to_numpy()
    records = []
    for (c,s), mat in norm_profiles.items():
        for cat in CLASS_ORDER:
            gm = cats == cat
            if not gm.any():
                continue
            prof = safe_nanmean(mat[gm], axis=0)
            for b, value in enumerate(prof):
                records.append({"celltype":target_celltype,"region":target_region,"axis":target_axis,
                                "condition":c,"sample":s,"category":cat,"bin":b,
                                "bin_center":BIN_CENTERS[b],"value":value,
                                "n_category_genes":int(gm.sum())})
    sample_df = pd.DataFrame(records, columns=["celltype","region","axis","condition","sample",
                                               "category","bin","bin_center","value","n_category_genes"])
    if sample_df.empty:
        raise ValueError(f"{prefix}: no high_to_low / low_to_high genes remain after filtering; check QC and its thresholds")
    summary = sample_df.groupby(["celltype","region","axis","condition","category","bin","bin_center"],
                                observed=True)["value"].agg(["mean","std","count"]).reset_index()
    summary["sem"] = summary["std"] / np.sqrt(summary["count"])
    n_high = int((cats == "high_to_low").sum())
    n_low = int((cats == "low_to_high").sum())
    cat_counts = {"celltype":target_celltype,"region":target_region,"axis":target_axis,
                  "n_all_genes":int(adata.n_vars),"n_selected_genes":int(selected.sum()),
                  "n_high_to_low":n_high,"n_low_to_high":n_low,
                  "n_non_gradient":int((cats == "non_gradient").sum()),
                  "n_cells":int(len(chosen)),"n_WT_samples":len(sample_names["WT"]),
                  "n_Heter_samples":len(sample_names["Heter"]),
                  "n_Homo_samples":len(sample_names["Homo"]),
                  "qc_enabled":bool(USE_WT_GENE_QC),"manual_genes":str(MANUAL_GENES)}
    return {"meta":selected_meta.assign(celltype=target_celltype,region=target_region,axis=target_axis),
            "audit":audit,"sample":sample_df,"summary":summary,
            "bin_counts":pd.DataFrame(count_records),"counts":cat_counts}


In [ ]:
def plot_celltype(target_celltype, combined):
    fig_w = LEFT_MM + len(AXIS_SPECS)*PANEL_W_MM + (len(AXIS_SPECS)-1)*PANEL_GAP_MM + RIGHT_MM
    fig_h = BOTTOM_MM + PANEL_H_MM + TOP_MM
    fig = plt.figure(figsize=(fig_w/25.4, fig_h/25.4))
    for i,(region,axis_name) in enumerate(AXIS_SPECS):
        ax = fig.add_axes([(LEFT_MM+i*(PANEL_W_MM+PANEL_GAP_MM))/fig_w,
                           BOTTOM_MM/fig_h, PANEL_W_MM/fig_w, PANEL_H_MM/fig_h])
        result = combined[(region,axis_name)]
        summary = result["summary"]
        n_high, n_low = result["counts"]["n_high_to_low"], result["counts"]["n_low_to_high"]
        for c in CONDITIONS:
            for cat in CLASS_ORDER:
                sub = summary.loc[(summary["condition"]==c)&(summary["category"]==cat)].sort_values("bin")
                if sub.empty: continue
                x = sub["bin_center"].to_numpy(float)
                y = sub["mean"].to_numpy(float)
                err = sub[ERROR_TYPE].to_numpy(float)
                ax.plot(x, y, color=CONDITION_COLORS[c], linestyle=CLASS_LINESTYLE[cat], lw=LINEWIDTH)
                ax.fill_between(x, y-err, y+err, color=CONDITION_COLORS[c], alpha=ERROR_ALPHA, linewidth=0)
        ax.axhline(1, color="0.65", linewidth=.45, linestyle=":")
        ax.set_xlim(0,100)
        if Y_LIMITS is not None: ax.set_ylim(*Y_LIMITS)
        ax.set_xticks([0,20,40,60,80,100])
        if Y_LIMITS == (0.5,1.5): ax.set_yticks([0.5,0.75,1.,1.25,1.5])
        ax.set_xlabel(f"{region}-{axis_name} axis (0–100)")
        if i==0: ax.set_ylabel("Relative expression\n(gene mean = 1)" if PROFILE_NORMALIZATION=="gene_mean1"
                                else "Mean norm_1e4 expression")
        ax.set_title(f"{region}-{axis_name} | H→L n={n_high}, L→H n={n_low}", pad=2)
        ax.tick_params(width=.45,length=2,pad=1)
        for sp in ax.spines.values(): sp.set_linewidth(.45)
    handles = [Line2D([0],[0],color=CONDITION_COLORS[c],lw=LINEWIDTH,label=c) for c in CONDITIONS]
    handles += [Line2D([0],[0],color="black",lw=LINEWIDTH,ls=CLASS_LINESTYLE[cat],label=CLASS_LABEL[cat])
                for cat in CLASS_ORDER]
    fig.legend(handles=handles, frameon=False, loc="upper center",bbox_to_anchor=(.5,1.005),
               ncol=5,handlelength=1.7,columnspacing=1.3)
    out_pdf = OUT_ROOT / f"{target_celltype}_5axes_gradient_class_profiles.pdf"
    try:
        fig.savefig(out_pdf, dpi=DPI, bbox_inches="tight", pad_inches=.01, transparent=True)
        if SHOW_FIGURES: plt.show()
    finally:
        plt.close(fig)
    print("Saved:",out_pdf)


In [ ]:
summary_all = []
for target in TARGET_CELLTYPES:
    combined = {}
    meta_list, audit_list, sample_list, summary_list, bin_list, count_list = [], [], [], [], [], []
    for region, axis_name in AXIS_SPECS:
        print(f"\n{target} | {region}-{axis_name}")
        r = analyze_axis(target,region,axis_name)
        combined[(region,axis_name)] = r
        meta_list.append(r["meta"])
        audit_list.append(r["audit"])
        sample_list.append(r["sample"])
        summary_list.append(r["summary"])
        bin_list.append(r["bin_counts"])
        count_list.append(r["counts"])
        print("  selected/high/low:",r["counts"]["n_selected_genes"],
              r["counts"]["n_high_to_low"],r["counts"]["n_low_to_high"])
    pd.concat(meta_list, ignore_index=True).to_csv(OUT_ROOT / f"{target}_5axes_gene_classes.csv",index=False)
    pd.concat(audit_list, ignore_index=True).to_csv(OUT_ROOT / f"{target}_5axes_gene_filter_QC.csv",index=False)
    pd.concat(sample_list, ignore_index=True).to_csv(OUT_ROOT / f"{target}_5axes_gradient_class_sample_profiles.csv",index=False)
    pd.concat(summary_list, ignore_index=True).to_csv(OUT_ROOT / f"{target}_5axes_gradient_class_profile_summary.csv",index=False)
    pd.concat(bin_list, ignore_index=True).to_csv(OUT_ROOT / f"{target}_5axes_sample_bin_cell_counts.csv",index=False)
    pd.DataFrame(count_list).to_csv(OUT_ROOT / f"{target}_5axes_class_counts_and_sample_QC.csv",index=False)
    plot_celltype(target,combined)
    summary_all.extend(count_list)
    del combined, meta_list, audit_list, sample_list, summary_list, bin_list
    gc.collect()
pd.DataFrame(summary_all).to_csv(OUT_ROOT / "_D1_D2_5axes_analysis_summary.csv",index=False)
print("\nALL DONE:", OUT_ROOT)


## Panel Q | Relative GO score heatmaps

Requires precomputed GO-score h5ad files and term metadata. GO gene-set construction and score generation are not included. D1 outputs use the suffix `*_01_condition_relative_heatmap.pdf`. The default `RELATIVE_VMAX=0.5` differs from the legend range of +/-0.4. Batch preflight requires D1 and D2 files for all five axes.


In [ ]:
# Imports and settings.
from pathlib import Path
import gc
import logging
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse


SOURCE_ROOT = Path(r"F:\BASSFISH_HD\Source_data")
SCORE_ROOT = SOURCE_ROOT / "term_scores"
GENE_CELL_H5AD = SOURCE_ROOT / "1228genes_x_y_axis.h5ad"
OUT_DIR = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene\go") / "GO_CV_aligned_relative_heatmaps_10axes"
OUT_DIR.mkdir(parents=True, exist_ok=True)


BATCH_AXES = [("d", "x"), ("d", "y"), ("n", "x"), ("n", "y"), ("o", "x")]
BATCH_CELLTYPES = ["D1", "D2"]

REQUIRE_ALL_INPUT_FILES = True

CONTINUE_ON_TASK_ERROR = True


CONDITIONS = ["WT", "Heter", "Homo"]
AXIS_COLUMNS = {"x": "region_manual_axis_0_100", "y": "region_manual_axis_y_0_100"}
N_BINS = 10


CV_MIN_VALID_BINS = 5
CV_MIN_POSITIVE_MEAN = 1e-8
CV_MIN_SAMPLES_FRACTION = 0.50
CV_REQUIRE_NONNEGATIVE_BINS = False
RELATIVE_RESPECT_CV_QC = True
RELATIVE_VMAX = 0.5
CV_VMAX = 1.0
DELTA_CV_MAX = 0.5
OUTPUT_LEGACY_ROW_Z = True
OUTPUT_RELATIVE_PER_SAMPLE = True
OUTPUT_RELATIVE_COMBINED = True
PRINT_EVERY_TASK = True
BIN_AGGREGATION = "cell_mean"
ZSCORE_VMAX = 1
CMAP = "RdBu_r"
STRICT_CELL_ID_MATCH = True
MIN_CELL_ID_MATCH_RATE = 0.95


AXES_WIDTH_MM = 10
AXES_HEIGHT_MM = 20
PANEL_GAP_MM = 3
COLORBAR_WIDTH_MM = 3
COLORBAR_GAP_MM = 2
LEFT_MARGIN_MM = 7
RIGHT_MARGIN_MM = 4
BOTTOM_MARGIN_MM = 14
TOP_MARGIN_MM = 8
DPI = 600
SHOW_FIGURES = False

plt.rcParams.update({
    "font.family": "Arial", "font.size": 7, "axes.labelsize": 7,
    "axes.titlesize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "pdf.fonttype": 42, "ps.fonttype": 42
})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)

print("GO term directory:", SCORE_ROOT)
print("Coordinate reference:", GENE_CELL_H5AD)
print("Output directory:", OUT_DIR)


In [ ]:
CT_MAP = {"D1": "MSN_D1", "D2": "MSN_D2", "MSN_D1": "MSN_D1", "MSN_D2": "MSN_D2"}


def locate_score_h5ad(region, celltype, axis):
    if region not in {"d", "n", "o"} or axis not in AXIS_COLUMNS:
        raise ValueError(f"Unsupported region/axis: {region}, {axis}")
    ct = CT_MAP.get(str(celltype).upper())
    if ct is None:
        raise ValueError(f"Unsupported celltype: {celltype}")
    filename = f"axis_{axis}_region_{region}_celltype_{ct}_representative_term_scores.h5ad"

    candidates = [
        SCORE_ROOT / "cell_level_representative_term_scores" / axis / f"region_{region}" / f"celltype_{ct}" / filename,
        SCORE_ROOT / axis / f"region_{region}" / f"celltype_{ct}" / filename,
        SCORE_ROOT / filename,
    ]
    found = [p for p in candidates if p.is_file()]
    if not found:
        if not SCORE_ROOT.is_dir():
            raise FileNotFoundError(f"GO term directory not found: {SCORE_ROOT}")
        found = sorted(SCORE_ROOT.rglob(filename))
    found = list(dict.fromkeys(p.resolve() for p in found))
    if not found:
        raise FileNotFoundError(f"GO score file {filename} not found under {SCORE_ROOT}")
    if len(found) != 1:
        raise ValueError(f"Multiple GO score files have the same name; selection is ambiguous: {found}")
    return found[0], ct


def load_coordinate_reference():
    if not GENE_CELL_H5AD.is_file():
        raise FileNotFoundError(f"Coordinate source file not found: {GENE_CELL_H5AD}")
    print("Reading coordinate reference:", GENE_CELL_H5AD)
    ref = sc.read_h5ad(GENE_CELL_H5AD, backed="r")
    try:
        if not ref.obs_names.is_unique:
            raise ValueError("obs_names is not unique in the coordinate-reference h5ad")
        required = list(AXIS_COLUMNS.values())
        missing = [c for c in required if c not in ref.obs.columns]
        if missing:
            raise KeyError(f"Coordinate reference is missing {missing}; available coordinate columns: {[c for c in ref.obs.columns if 'axis' in c]}")
        copy_cols = required + [c for c in ["sample", "condition"] if c in ref.obs.columns]
        ref_obs = ref.obs[copy_cols].copy()
    finally:
        file = getattr(ref, "file", None)
        if file is not None:
            file.close()
    return ref_obs


def attach_axis_by_cell_id(adata, ref_obs, axis_col, source_name):
    if not adata.obs_names.is_unique:
        raise ValueError(f"{source_name}: GO score obs_names is not unique")
    loc = ref_obs.index.get_indexer(adata.obs_names)
    found = loc >= 0
    match_rate = float(found.mean()) if len(found) else 0.0
    if STRICT_CELL_ID_MATCH and not np.all(found):
        missing = adata.obs_names[~found]
        raise ValueError(f"{source_name}: {len(missing)} cell IDs cannot be matched to coordinates; examples: {list(missing[:10])}")
    if match_rate < MIN_CELL_ID_MATCH_RATE:
        raise ValueError(f"{source_name}: cell match rate {match_rate:.2%} is below {MIN_CELL_ID_MATCH_RATE:.0%}")

    aligned = ref_obs.reindex(adata.obs_names)
    for field in ["sample", "condition"]:
        if field in adata.obs and field in aligned:
            a = adata.obs[field].astype("string").str.strip().to_numpy()
            b = aligned[field].astype("string").str.strip().to_numpy()
            mismatch = np.asarray(found) & ~(pd.Series(a).fillna("<NA>").to_numpy() == pd.Series(b).fillna("<NA>").to_numpy())
            if mismatch.any():
                bad = np.flatnonzero(mismatch)[:10]
                raise ValueError(f"{source_name}: {field} differs in {int(mismatch.sum())} matched cells; example IDs: {list(adata.obs_names[bad])}")

    coords = pd.to_numeric(aligned[axis_col], errors="coerce").to_numpy(float)
    info = {"h5ad": str(source_name), "n_cells": adata.n_obs,
            "n_matched_cell_ids": int(found.sum()), "cell_id_match_rate": match_rate,
            "n_finite_axis": int(np.isfinite(coords).sum()),
            "n_axis_0_100": int((np.isfinite(coords) & (coords >= 0) & (coords <= 100)).sum())}
    return coords, info


In [ ]:
def bin_axis_0_100(axis, n_bins=N_BINS):
    axis = np.asarray(axis, dtype=float)
    valid = np.isfinite(axis) & (axis >= 0) & (axis <= 100)
    bins = np.full(len(axis), -1, dtype=int)

    bins[valid] = np.minimum(np.floor(axis[valid] / (100 / n_bins)).astype(int), n_bins - 1)
    return bins, valid


def mean_rows(X, idx):
    if len(idx) == 0:
        return np.full(X.shape[1], np.nan)
    z = X[idx]
    return np.asarray(z.mean(axis=0)).ravel() if sparse.issparse(z) else np.nanmean(np.asarray(z), axis=0)


def calc_profiles(adata, axis, condition, sample, n_bins=N_BINS):
    bins, valid = bin_axis_0_100(axis, n_bins)
    groups = {}
    count_records = []
    for c in CONDITIONS:
        mask = valid & (condition == c)
        pooled = np.full((adata.n_vars, n_bins), np.nan, dtype=float)
        for b in range(n_bins):
            idx = np.flatnonzero(mask & (bins == b))
            if len(idx):
                pooled[:, b] = mean_rows(adata.X, idx)
            count_records.append({"condition": c, "sample": "ALL", "bin": b + 1,
                                  "bin_left": b * (100 / n_bins), "bin_right": (b+1) * (100 / n_bins),
                                  "n_cells": int(len(idx)), "aggregation": "pooled_cells"})
        if BIN_AGGREGATION == "cell_mean":
            groups[c] = pooled
        elif BIN_AGGREGATION == "sample_mean":
            mats = []
            for s in sorted(pd.unique(sample[mask])):
                smask = mask & (sample == s)
                mat = np.full((adata.n_vars, n_bins), np.nan, dtype=float)
                for b in range(n_bins):
                    idx = np.flatnonzero(smask & (bins == b))
                    if len(idx):
                        mat[:, b] = mean_rows(adata.X, idx)
                mats.append(mat)
            if mats:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    groups[c] = np.nanmean(np.stack(mats, axis=0), axis=0)
            else:
                groups[c] = np.full((adata.n_vars, n_bins), np.nan, dtype=float)
        else:
            raise ValueError("BIN_AGGREGATION must be cell_mean or sample_mean")
        for s in sorted(pd.unique(sample[mask])):
            for b in range(n_bins):
                cnt = int(np.sum(mask & (sample == s) & (bins == b)))
                count_records.append({"condition": c, "sample": str(s), "bin": b + 1,
                                      "bin_left": b * (100 / n_bins), "bin_right": (b+1) * (100 / n_bins),
                                      "n_cells": cnt, "aggregation": "one_sample"})
    return groups, pd.DataFrame(count_records), bins, valid


def get_original_term_order(adata):
    if not adata.var_names.is_unique:
        raise ValueError("GO term var_names is not unique; ordering is ambiguous")
    required = ["term_name", "spatial_category", "pattern_strength"]
    missing = [k for k in required if k not in adata.var.columns]
    if missing:
        raise KeyError(f"GO score h5ad var is missing required metadata: {missing}; term ordering cannot be reconstructed")
    term_meta = adata.var.copy()
    term_meta["term_id"] = adata.var_names.astype(str)
    term_meta["term_name"] = term_meta["term_name"].astype(str)
    category = term_meta["spatial_category"].astype(str)
    strength = pd.to_numeric(term_meta["pattern_strength"], errors="coerce").fillna(0).to_numpy()
    signed_strength = np.zeros(len(term_meta), dtype=float)
    signed_strength[category.eq("high_to_low").to_numpy()] = strength[category.eq("high_to_low").to_numpy()]
    signed_strength[category.eq("low_to_high").to_numpy()] = -strength[category.eq("low_to_high").to_numpy()]
    term_meta["signed_pattern_strength"] = signed_strength

    term_meta = term_meta.sort_values(["signed_pattern_strength", "term_name"], ascending=[False, True])
    order = adata.var_names.get_indexer(term_meta.index)
    if np.any(order < 0):
        raise RuntimeError("Term-order indexing failed")
    return term_meta, order


def row_zscore(x):

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(x, axis=1, keepdims=True)
        sd = np.nanstd(x, axis=1, ddof=0, keepdims=True)
    z = np.divide(x - mean, sd, out=np.zeros_like(x, dtype=float), where=np.isfinite(sd) & (sd > 0))

    z[np.all(~np.isfinite(x), axis=1), :] = np.nan
    return z


def term_class_separators(term_meta):
    signed = term_meta["signed_pattern_strength"].to_numpy(float)
    zeros = np.where(np.isclose(signed, 0))[0]
    if len(zeros):
        return [zeros[0] - 0.5, zeros[-1] + 0.5]
    neg = np.where(signed < 0)[0]
    return [neg[0] - 0.5] if len(neg) else []


In [ ]:
def plot_go_heatmap(plot_mat, term_meta, region, ct, axis, out_path):
    fig_width_mm = (LEFT_MARGIN_MM + 3*AXES_WIDTH_MM + 2*PANEL_GAP_MM +
                    COLORBAR_GAP_MM + COLORBAR_WIDTH_MM + RIGHT_MARGIN_MM)
    fig_height_mm = BOTTOM_MARGIN_MM + AXES_HEIGHT_MM + TOP_MARGIN_MM
    fig = plt.figure(figsize=(fig_width_mm / 25.4, fig_height_mm / 25.4))
    fig.text(LEFT_MARGIN_MM / fig_width_mm, 1-(TOP_MARGIN_MM*.3)/fig_height_mm,
             f"{ct} | {region}-{axis} axis | Original GO row z-score",fontsize=8,va="center")
    axes = []
    x_mm = LEFT_MARGIN_MM
    for _ in CONDITIONS:
        axes.append(fig.add_axes([x_mm/fig_width_mm, BOTTOM_MARGIN_MM/fig_height_mm,
                                 AXES_WIDTH_MM/fig_width_mm, AXES_HEIGHT_MM/fig_height_mm]))
        x_mm += AXES_WIDTH_MM + PANEL_GAP_MM
    cb_left_mm = LEFT_MARGIN_MM + 3*AXES_WIDTH_MM + 2*PANEL_GAP_MM + COLORBAR_GAP_MM
    cax = fig.add_axes([cb_left_mm/fig_width_mm, BOTTOM_MARGIN_MM/fig_height_mm,
                        COLORBAR_WIDTH_MM/fig_width_mm, AXES_HEIGHT_MM/fig_height_mm])
    bin_labels = [f"{i*10}–{(i+1)*10}" for i in range(N_BINS)]
    separators = term_class_separators(term_meta)
    axis_col = AXIS_COLUMNS[axis]
    try:
        im = None
        for ax, c in zip(axes, CONDITIONS):

            im = ax.imshow(plot_mat[c], aspect="auto", interpolation="nearest",
                           cmap=CMAP, vmin=-ZSCORE_VMAX, vmax=ZSCORE_VMAX,
                           rasterized=False)
            ax.set_xticks(range(N_BINS))
            ax.set_xticklabels(bin_labels, rotation=45, ha="right")
            ax.set_xlabel(axis_col)
            ax.set_title(c, pad=2)
            ax.set_yticks([])
            for y in separators:
                ax.axhline(y, color="black", lw=.8)
            ax.tick_params(axis="x", width=.5, length=2, pad=1, labelsize=7)
            for spine in ax.spines.values():
                spine.set_linewidth(.5)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Row z-score", labelpad=2)
        cbar.ax.tick_params(labelsize=7, width=.5, length=2, pad=1)
        cbar.outline.set_linewidth(.5)
        fig.savefig(out_path, dpi=DPI, bbox_inches="tight", pad_inches=.01, transparent=True)
        if SHOW_FIGURES:
            plt.show()
    finally:
        plt.close(fig)
    return out_path


In [ ]:
def nanmean_no_warn(A, axis):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return np.nanmean(A, axis=axis)


def go_score_validity(bin_means):
    """Check GO score validity for relative profiles and CV."""
    a = np.asarray(bin_means, float)
    n_valid = np.isfinite(a).sum(axis=1)
    mu = nanmean_no_warn(a, axis=1)
    sd = np.nanstd(a, axis=1, ddof=0)
    valid = ((n_valid >= CV_MIN_VALID_BINS) & np.isfinite(mu) &
             (mu > CV_MIN_POSITIVE_MEAN) & np.isfinite(sd))
    if CV_REQUIRE_NONNEGATIVE_BINS:
        valid &= np.all((~np.isfinite(a)) | (a >= 0), axis=1)
    return valid, n_valid, mu, sd


def go_relative_and_cv(bin_means):
    """Compute relative GO profiles and CV without filling missing values."""
    a = np.asarray(bin_means, float)
    valid, n_valid, mu, sd = go_score_validity(a)
    rel = np.full(a.shape, np.nan, float)
    cv = np.full(a.shape[0], np.nan, float)
    idx = np.flatnonzero(valid)
    if len(idx):
        rel[idx] = a[idx] / mu[idx,None] - 1.0
        cv[idx] = sd[idx] / mu[idx]

        check = np.nanstd(rel[idx], axis=1, ddof=0)
        if not np.allclose(check, cv[idx], rtol=1e-8, atol=1e-9):
            raise AssertionError('Relative profiles and CV are inconsistent; check numeric inputs')
    return rel, cv, valid, n_valid, mu


def calc_sample_bin_means(ad, bins, valid_axis, condition, sample, order):
    """Compute ten-bin GO score means separately for each animal."""
    results = {}
    count_records = []
    for c in CONDITIONS:
        names = sorted(str(s) for s in pd.unique(sample[valid_axis & (condition == c)]))
        if not names:
            raise ValueError(f'Condition {c} has no valid animals for plotting or CV calculation')
        for name in names:
            sel = valid_axis & (condition == c) & (sample == name)
            mat = np.full((ad.n_vars, N_BINS), np.nan, dtype=float)
            for k in range(N_BINS):
                ii = np.flatnonzero(sel & (bins == k))
                if len(ii):
                    mat[:,k] = mean_rows(ad.X, ii)
                count_records.append(dict(condition=c, sample=name, bin=k+1, n_cells=len(ii),
                                          bin_left=k*100./N_BINS, bin_right=(k+1)*100./N_BINS))
            results[(c,name)] = mat[order]
    return results, pd.DataFrame(count_records)


def make_cv_tables(meta, sample_raw, prefix):
    """Summarize per-animal GO term CVs and genotype differences."""
    term_ids = meta['term_id'].to_numpy(str)
    cv_by_sample = {}; rel_by_sample = {}
    rows = []
    for (c,s),mat in sample_raw.items():
        rel,cv,valid,n_valid,mu = go_relative_and_cv(mat)
        cv_by_sample[(c,s)] = cv
        rel_by_sample[(c,s)] = rel
        for j,tid in enumerate(term_ids):
            rows.append(dict(term_id=tid, term_name=meta['term_name'].iat[j],
                             spatial_category=meta['spatial_category'].iat[j],
                             condition=c, sample=s, cv=cv[j],
                             score_mean=mu[j], n_valid_bins=int(n_valid[j]),
                             pass_sample_cv_qc=bool(valid[j])))
    sample_table = pd.DataFrame(rows)
    cvs = {}; nvalid = {}; c_records = []; condition_relative = {}
    for c in CONDITIONS:
        keys = [k for k in sample_raw if k[0]==c]
        cv_stack = np.stack([cv_by_sample[k] for k in keys])
        rel_stack = np.stack([rel_by_sample[k] for k in keys])
        n = np.isfinite(cv_stack).sum(axis=0)
        required=max(1,int(np.ceil(len(keys)*CV_MIN_SAMPLES_FRACTION)))
        cmean=nanmean_no_warn(cv_stack, axis=0)
        cmean[n<required] = np.nan

        relative = nanmean_no_warn(rel_stack, axis=0)
        relative[n<required,:] = np.nan
        cvs[c]=cmean; nvalid[c]=n;condition_relative[c]=relative
        for j,tid in enumerate(term_ids):
            vals=cv_stack[:,j]; vals=vals[np.isfinite(vals)]
            c_records.append(dict(term_id=tid,term_name=meta['term_name'].iat[j], condition=c,
                                  cv_animal_mean=cmean[j],n_animals_valid=int(n[j]),
                                  n_animals_total=len(keys),n_animals_required=required,
                                  cv_sd=np.std(vals,ddof=1) if len(vals)>=2 else np.nan,
                                  cv_sem=np.std(vals,ddof=1)/np.sqrt(len(vals)) if len(vals)>=2 else np.nan))
    paired = np.isfinite(cvs['WT']) & np.isfinite(cvs['Heter']) & np.isfinite(cvs['Homo'])
    delta={c:np.where(paired,cvs[c]-cvs['WT'],np.nan) for c in ['Heter','Homo']}
    summary=pd.DataFrame([dict(term_id=tid, term_name=meta['term_name'].iat[j],
            spatial_category=meta['spatial_category'].iat[j],
            WT_CV=cvs['WT'][j],Heter_CV=cvs['Heter'][j],Homo_CV=cvs['Homo'][j],
            Heter_minus_WT=delta['Heter'][j],Homo_minus_WT=delta['Homo'][j],
            paired_all_conditions=bool(paired[j]),
            WT_n_animals_valid=int(nvalid['WT'][j]),Heter_n_animals_valid=int(nvalid['Heter'][j]),
            Homo_n_animals_valid=int(nvalid['Homo'][j]))
            for j,tid in enumerate(term_ids)])
    sample_table.to_csv(OUT_DIR/f'{prefix}_GO_term_CV_per_sample.csv',index=False)
    pd.DataFrame(c_records).to_csv(OUT_DIR/f'{prefix}_GO_term_CV_per_condition.csv',index=False)
    summary.to_csv(OUT_DIR/f'{prefix}_GO_term_CV_delta_WT.csv',index=False)
    if (~paired).any():
        print(f'  GO CV: {int(paired.sum())}/{len(paired)} terms have complete WT/Heter/Homo pairing; '
              f'other terms are marked missing (e.g., nonpositive mean GO score or insufficient valid bins)')
    if not paired.any():
        warnings.warn(f'{prefix}: no terms have CV values paired across all three conditions; check that GO scores are positive and suitable for ratios. Do not shift scores arbitrarily.')
    return rel_by_sample, condition_relative, cvs, delta, summary


def export_score_profile_matrices(meta, sample_raw, rel_by_sample, condition_relative, prefix):
    base_cols=['term_id','term_name','spatial_category','pattern_strength']
    fixed=meta[base_cols].reset_index(drop=True)
    def long_table(data):
        records=[]
        for (c,s),arr in data.items():
            for j,tid in enumerate(meta['term_id'].astype(str)):
                for k in range(N_BINS):
                    records.append((tid,c,s,k+1,float(arr[j,k])))
        return pd.DataFrame(records,columns=['term_id','condition','sample','bin','score'])

    long_table(sample_raw).to_csv(OUT_DIR/f'{prefix}_GO_term_bin_scores_per_sample.csv',index=False)
    rel_all=long_table(rel_by_sample).rename(columns={'score':'relative_score'})
    rel_all.to_csv(OUT_DIR/f'{prefix}_GO_term_relative_per_sample.csv',index=False)
    mats=[]
    for c in CONDITIONS:
        arr=condition_relative[c]
        mats.append(pd.DataFrame(arr,columns=[f'{c}_bin_{i+1}' for i in range(N_BINS)]))
    pd.concat([fixed]+mats,axis=1).to_csv(OUT_DIR/f'{prefix}_GO_term_relative_condition_animal_mean.csv',index=False)


def plot_go_panels(panels, meta, region, ct, axis, out_path, title_suffix, vmin=None, vmax=None):
    """Plot GO profiles using fixed term ordering and color limits."""
    count=len(panels)
    PWIDTH=12.; PGAP=2.; GROUP_GAP=1.
    left=9.; right=5.; top=12.; bottom=16.; cbar_w=3.;cbar_gap=5.; H=31.
    positions=[]; x=left; prev=None
    for title,group,arr in panels:
        if prev is not None and group!=prev:
            x += GROUP_GAP
        positions.append(x)
        x += PWIDTH+PGAP
        prev=group
    last=positions[-1]+PWIDTH
    fw=last+cbar_gap+cbar_w+right
    fh=bottom+H+top
    fig=plt.figure(figsize=(fw/25.4,fh/25.4))
    cmap=plt.get_cmap('RdBu_r').copy();cmap.set_bad('white')
    low=(-RELATIVE_VMAX if vmin is None else vmin)
    hi=(RELATIVE_VMAX if vmax is None else vmax)
    try:
        im=None
        for k,((title,group,arr),xp) in enumerate(zip(panels,positions)):
            ax=fig.add_axes([xp/fw,bottom/fh,PWIDTH/fw,H/fh])
            im=ax.imshow(np.ma.masked_invalid(arr),interpolation='nearest',aspect='auto',
                         cmap=cmap,vmin=low,vmax=hi,rasterized=False)
            ax.set_title(str(title),fontsize=6.5,pad=2)
            ax.set_xticks(np.arange(N_BINS))
            ax.set_xticklabels([str(i+1) for i in range(N_BINS)],fontsize=6)
            ax.set_xlabel('Spatial bin (0–100)',fontsize=6.5)
            ax.set_yticks([])
            for sep in term_class_separators(meta):
                ax.axhline(sep,c='0.4',lw=.5)
            if k and panels[k][1]!=panels[k-1][1]:
                ax.axvline(-.5,lw=.8,c='0.2')
            for sp in ax.spines.values():sp.set_linewidth(.4)
        cbax=fig.add_axes([(last+cbar_gap)/fw,bottom/fh,cbar_w/fw,H/fh])
        cb=fig.colorbar(im,cax=cbax);cb.set_label('GO score / own bin-mean − 1',fontsize=6.5)
        cb.ax.tick_params(labelsize=6,width=.5,length=2)
        fig.text(left/fw, 1-(top*.28)/fh,
                 f'{ct} | {region}-{axis} axis | {title_suffix}',fontsize=8,va='center')
        fig.savefig(out_path,dpi=DPI,bbox_inches='tight',pad_inches=.02,facecolor='white')
    finally:
        plt.close(fig)
    print('  Saved:',out_path.name)


def plot_go_cv_heatmap(meta,cvs,delta,region,ct,axis,out_path):
    """Plot genotype mean GO term CVs and changes from WT."""
    mat_cv=np.column_stack([cvs[c] for c in CONDITIONS])
    mat_delta=np.column_stack([delta['Heter'],delta['Homo']])
    n=len(meta)
    fig=plt.figure(figsize=(4.8,max(2.5,min(10.,n*.019+1.2))))
    try:
        ax1=fig.add_axes([.09,.10,.37,.80]);ax2=fig.add_axes([.59,.10,.24,.80])
        c1=plt.get_cmap('Reds').copy();c1.set_bad('white')
        c2=plt.get_cmap('RdBu_r').copy();c2.set_bad('white')
        im1=ax1.imshow(np.ma.masked_invalid(mat_cv),cmap=c1,vmin=0,vmax=CV_VMAX,
                       aspect='auto',interpolation='nearest',rasterized=False)
        im2=ax2.imshow(np.ma.masked_invalid(mat_delta),cmap=c2,vmin=-DELTA_CV_MAX,vmax=DELTA_CV_MAX,
                       aspect='auto',interpolation='nearest',rasterized=False)
        ax1.set_xticks([0,1,2],CONDITIONS); ax2.set_xticks([0,1],['Heter−WT','Homo−WT'])
        for ax in [ax1,ax2]:
            ax.set_yticks([]);ax.tick_params(axis='x',length=0,labelsize=6,labelrotation=25)
            for s in ax.spines.values():s.set_linewidth(.4)
        if n<=45:
            ax1.set_yticks(range(n));ax1.set_yticklabels(meta['term_name'],fontsize=4.2)
        ca1=fig.add_axes([.47,.10,.018,.80]);ca2=fig.add_axes([.845,.10,.018,.80])
        cbar1=fig.colorbar(im1,cax=ca1);cbar1.set_label('CV (animal mean)',fontsize=7)
        cbar2=fig.colorbar(im2,cax=ca2);cbar2.set_label('ΔCV',fontsize=7)
        fig.text(.09,.965,f'{ct} | {region}-{axis} axis | GO term CV / ΔCV (per animal)',fontsize=8,va='center')
        fig.savefig(out_path,dpi=DPI,bbox_inches='tight',pad_inches=.03,facecolor='white')
    finally:
        plt.close(fig)
    print('  Saved:',out_path.name)


In [ ]:
def analyze_one_go_task(region, celltype, axis, ref_obs):
    path, ct = locate_score_h5ad(region, celltype, axis)
    print(f'\nGO term: region={region} | celltype={ct} | axis={axis} | {path}',flush=True)
    score_ad=sc.read_h5ad(path)
    try:
        for field in ['sample','condition']:
            if field not in score_ad.obs:
                raise KeyError(f'{path.name}: missing obs[{field!r}]')
        if score_ad.n_obs==0 or score_ad.n_vars==0:
            raise ValueError('GO score h5ad contains zero cells or zero GO terms')
        coords,audit=attach_axis_by_cell_id(score_ad,ref_obs,AXIS_COLUMNS[axis],path)
        condraw=score_ad.obs['condition'].astype('string').str.strip().to_numpy(dtype=str,na_value='<NA>')
        match={c.lower():c for c in CONDITIONS}
        condition=np.asarray([match.get(c.lower(),'<OTHER>') for c in condraw],dtype=object)
        sample=score_ad.obs['sample'].astype('string').str.strip().to_numpy(dtype=str,na_value='<NA>')
        bins,valid=bin_axis_0_100(coords)
        missing=set(CONDITIONS)-set(condition[valid])
        if missing:raise ValueError(f'Conditions missing from valid coordinates: {sorted(missing)}')
        term_meta,order=get_original_term_order(score_ad)
        prefix=f'axis_{axis}_region_{region}_celltype_{ct}'

        old_raw,count_df,_,_=calc_profiles(score_ad,coords,condition,sample)
        old_raw={c:old_raw[c][order] for c in CONDITIONS}
        old_z={c:row_zscore(old_raw[c]) for c in CONDITIONS}
        bin_labels=[f'{i*10}–{(i+1)*10}' for i in range(N_BINS)]
        meta_cols=[v for v in ['term_id','term_name','spatial_category','pattern_strength',
                   'signed_pattern_strength','directional_r','wt_mean_profile_r',
                   'gene_count','n_genes_used','genes_used'] if v in term_meta]
        def save_matrix(mats,basename):
            pd.concat([term_meta[meta_cols]]+
               [pd.DataFrame(mats[c],index=term_meta.index,
                 columns=[f'{c}_bin_{k+1}_{bin_labels[k]}' for k in range(N_BINS)])
                for c in CONDITIONS],axis=1).to_csv(OUT_DIR/f'{prefix}_{basename}.csv',index=False)
        save_matrix(old_raw,'legacy_raw_mean')
        save_matrix(old_z,'legacy_row_zscore')
        count_df.to_csv(OUT_DIR/f'{prefix}_bin_cell_counts_QC.csv',index=False)
        term_meta[meta_cols].to_csv(OUT_DIR/f'{prefix}_term_order_and_metadata.csv',index=False)
        if OUTPUT_LEGACY_ROW_Z:
            legacy_path=OUT_DIR/f'{prefix}_00_legacy_row_zscore.pdf'
            plot_go_heatmap(old_z,term_meta,region,ct,axis,legacy_path)
            print('  Saved:',legacy_path.name)

        sample_raw,counts_per_sample=calc_sample_bin_means(score_ad,bins,valid,condition,sample,order)
        counts_per_sample.to_csv(OUT_DIR/f'{prefix}_sample_bin_counts_QC.csv',index=False)
        rel_sample,rel_cond,cvs,delta,cv_summary=make_cv_tables(term_meta,sample_raw,prefix)
        export_score_profile_matrices(term_meta,sample_raw,rel_sample,rel_cond,prefix)
        panels_cond=[(c,c,rel_cond[c]) for c in CONDITIONS]
        ordered_keys=[k for c in CONDITIONS for k in sample_raw if k[0]==c]
        panels_sample=[(s,c,rel_sample[(c,s)]) for c,s in ordered_keys]

        plot_go_panels(panels_cond,term_meta,region,ct,axis,
            OUT_DIR/f'{prefix}_01_condition_relative_heatmap.pdf',
            'GO relative score | condition mean of animal-normalized profiles')

        if OUTPUT_RELATIVE_PER_SAMPLE:
            plot_go_panels(panels_sample,term_meta,region,ct,axis,
                OUT_DIR/f'{prefix}_02_per_sample_relative_heatmap.pdf',
                'GO relative score | individual samples')

        if OUTPUT_RELATIVE_COMBINED:
            combo=[(f'Pooled {c}','Pooled',rel_cond[c]) for c in CONDITIONS]+panels_sample
            plot_go_panels(combo,term_meta,region,ct,axis,
                OUT_DIR/f'{prefix}_03_condition_plus_samples_relative_heatmap.pdf',
                'GO relative score | 3 conditions + individual samples')

        cv_pdf=OUT_DIR/f'{prefix}_04_GO_term_CV_and_delta_CV_heatmap.pdf'
        plot_go_cv_heatmap(term_meta,cvs,delta,region,ct,axis,cv_pdf)
        matched=int(cv_summary['paired_all_conditions'].sum())
        audit.update({'region':region,'celltype':ct,'axis':axis,'n_terms':score_ad.n_vars,
                      'n_samples':len(ordered_keys),'n_paired_cv_terms':matched,
                      'n_score_mean_nonpositive_or_invalid':int((~np.isfinite(cv_summary['WT_CV'])).sum()),
                      'bin_aggregation_legacy':BIN_AGGREGATION,
                      'cv_statistic':'mean of per-sample SD/mean',
                      'relative_formula':'E_bin / sample_term_bin_mean - 1',
                      'title_prefix':prefix,'main_pdf':str(cv_pdf)})
        print(f'  Done {prefix}: {len(ordered_keys)} samples, {matched}/{len(term_meta)} paired CV terms')
        return audit
    finally:
        del score_ad
        gc.collect()


In [ ]:
# Imports and settings.
import traceback

tasks = [(reg, ct, ax) for reg, ax in BATCH_AXES for ct in BATCH_CELLTYPES]
if len(tasks) != len(set(tasks)):
    raise ValueError("BATCH_AXES / BATCH_CELLTYPES contains duplicate tasks")
if not tasks:
    raise ValueError("The batch task list is empty")
print(f"\nBatch size = {len(tasks)}; axes = {BATCH_AXES}; cell types = {BATCH_CELLTYPES}")


preflight = []
locations = {}
for region, celltype, axis in tasks:
    rec = {"region": region, "celltype": celltype, "axis": axis,
           "task": f"{region}-{axis}_{celltype}"}
    try:
        p, ct = locate_score_h5ad(region, celltype, axis)
        locations[(region, celltype, axis)] = p
        rec.update({"status": "found", "score_h5ad": str(p), "problem": ""})
    except Exception as e:
        rec.update({"status": "missing_or_ambiguous", "score_h5ad": "", "problem": str(e)})
    preflight.append(rec)
preflight_df = pd.DataFrame(preflight)
preflight_path = OUT_DIR / "_GO_term_batch_input_preflight.csv"
preflight_df.to_csv(preflight_path, index=False, encoding="utf-8-sig")
print("\nInput preflight:")
print(preflight_df[["task", "status", "score_h5ad"]].to_string(index=False))
print("Saved:", preflight_path)

missing = preflight_df[preflight_df["status"] != "found"]
if REQUIRE_ALL_INPUT_FILES and len(missing):
    print("\nMissing/ambiguous GO score files:")
    print(missing[["task", "problem"]].to_string(index=False))
    raise FileNotFoundError(f"{len(missing)}/{len(tasks)} GO score files are missing or duplicated; check term_scores and the preflight CSV. No partial analysis was started.")

ref_obs = load_coordinate_reference()
records = []
manifest_path = OUT_DIR / "_GO_term_batch_execution_manifest.csv"
for i, (region, celltype, axis) in enumerate(tasks, start=1):
    taskname = f"{region}-{axis}_{celltype}"
    print(f"\n{'='*56}\n[{i}/{len(tasks)}] {taskname}\n{'='*56}", flush=True)
    try:
        if (region, celltype, axis) not in locations:
            raise FileNotFoundError(f"Not found during preflight: {taskname}")
        audit = analyze_one_go_task(region, celltype, axis, ref_obs)
        records.append({"task": taskname, "status": "success", "error": "", **audit})
    except Exception as e:
        print(f"[FAILED] {taskname}: {type(e).__name__}: {e}", flush=True)
        traceback.print_exc(limit=4)
        records.append({"task": taskname, "status": "error", "error": f"{type(e).__name__}: {e}",
                        "region": region, "celltype": celltype, "axis": axis,
                        "h5ad": str(locations.get((region, celltype, axis), ""))})
        if not CONTINUE_ON_TASK_ERROR:
            pd.DataFrame(records).to_csv(manifest_path, index=False, encoding="utf-8-sig")
            raise
    pd.DataFrame(records).to_csv(manifest_path, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame(records)
summary_df.to_csv(OUT_DIR / "_GO_term_axis_analysis_summary.csv", index=False, encoding="utf-8-sig")
success = int((summary_df["status"] == "success").sum())
failed = len(tasks) - success
print(f"\nBatch finished: {success}/{len(tasks)} successes; {failed} errors")
print("Manifest:", manifest_path)
print("Output:", OUT_DIR)
if failed:
    bad = summary_df.loc[summary_df["status"] != "success", ["task", "error"]]
    print(bad.to_string(index=False))
    raise RuntimeError(f"GO term batch has {failed} failed tasks; successful results and the error report were saved.")
print("ALL GO TERM AXIS PDFS GENERATED")
